# LeetCode 305 — Number of Islands II bằng Python (Hard)

> **Lưu ý: đây là bài Premium** (bị khóa trên LeetCode, cần tài khoản trả phí mới nộp được). Phần phát biểu đề, ví dụ, constraints và signature dưới đây viết theo đúng bản LeetCode. Nếu bạn không có Premium, cứ học bình thường: toàn bộ code trong notebook tự kiểm tra bằng `assert` và bằng một "trọng tài" chạy chậm nhưng chắc chắn đúng.

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def numIslands2(self, m: int, n: int, positions: List[List[int]]) -> List[int]:

```

Mục tiêu của bài học:

1. **Ôn lại bài #200 Number of Islands từ căn bản**: lưới 2D là một đồ thị, mỗi ô có 4 hàng xóm, dùng **BFS/DFS loang** (flood fill) để đếm **thành phần liên thông** (connected component — một nhóm ô đất nối với nhau).
2. Viết cách "ai cũng nghĩ ra": sau mỗi lần thêm đất thì **đếm lại toàn bộ lưới** — và đo cho thấy vì sao nó quá chậm (`O(k·m·n)`).
3. Rút ra **quan sát then chốt**: thêm một ô chỉ làm thay đổi **cục bộ** — *số đảo mới = số đảo cũ + 1 − (số đảo khác nhau chạm vào ô đó)*.
4. Cách trung gian: **gán nhãn đảo** và khi gộp thì **đổi nhãn phần nhỏ** bằng BFS (kỹ thuật "gộp nhỏ vào lớn" — small-to-large).
5. Học **Union-Find / DSU** (Disjoint Set Union — cấu trúc "các tập rời nhau") **từ con số 0**: `parent`, `find`, `union`; rồi hai tối ưu **path compression** (nén đường đi) và **union by size/rank** (gộp theo kích thước/hạng), có **đo độ cao cây** trước và sau tối ưu.
6. Áp dụng DSU cho bài: `count += 1`, rồi với mỗi hàng xóm là đất, **nếu `union` thành công thì `count -= 1`**; trace bảng `parent` từng bước; xử lý **ô bị thêm trùng**; biến thể **DSU dùng `dict`** chỉ lưu ô đất.
7. Test chéo mọi cách với "trọng tài" trên dữ liệu ngẫu nhiên có `random.seed`, nắm lỗi thường gặp và nhận dạng pattern **"kết nối được thêm dần, hỏi số nhóm"**.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một lưới 2D nhị phân `grid` rỗng kích thước `m x n`. Lưới biểu diễn một bản đồ, trong đó `0` là **nước** và `1` là **đất**. Ban đầu mọi ô của `grid` đều là nước (tức là toàn số `0`).
>
> Ta có thể thực hiện thao tác **thêm đất** (add land): biến nước tại một vị trí thành đất. Cho mảng `positions` trong đó `positions[i] = [ri, ci]` là vị trí `(ri, ci)` mà ta thực hiện thao tác thứ `i`.
>
> Trả về mảng số nguyên `answer` trong đó `answer[i]` là **số đảo** sau khi biến ô `(ri, ci)` thành đất.
>
> Một **đảo** (island) được bao quanh bởi nước và được tạo thành bằng cách nối các ô đất kề nhau **theo chiều ngang hoặc chiều dọc**. Có thể giả sử bốn cạnh của lưới đều được bao quanh bởi nước.

**Example 1**

```text
Input: m = 3, n = 3, positions = [[0,0],[0,1],[1,2],[2,1]]
Output: [1,1,2,3]
Explanation:
Ban đầu, lưới 2D toàn nước.
- Thao tác #1: addLand(0, 0) biến nước ở grid[0][0] thành đất. Ta có 1 đảo.
- Thao tác #2: addLand(0, 1) biến nước ở grid[0][1] thành đất. Vẫn có 1 đảo.
- Thao tác #3: addLand(1, 2) biến nước ở grid[1][2] thành đất. Ta có 2 đảo.
- Thao tác #4: addLand(2, 1) biến nước ở grid[2][1] thành đất. Ta có 3 đảo.
```

**Example 2**

```text
Input: m = 1, n = 1, positions = [[0,0]]
Output: [1]
```

### Constraints (ràng buộc) và ý nghĩa

```text
1 <= m, n, positions.length <= 10^4
1 <= m * n <= 10^4
positions[i].length == 2
0 <= ri < m
0 <= ci < n
```

**Follow up**: Bạn có giải được với độ phức tạp thời gian `O(k log(mn))` không, trong đó `k == positions.length`?

Từ nay ta đặt tên:

- `k = len(positions)` — số thao tác thêm đất, tối đa **10 000**.
- `N = m * n` — tổng số ô của lưới, tối đa **10 000**.

| Ràng buộc | Ý nghĩa cho lời giải |
|---|---|
| `m * n <= 10^4` | Lưới nhỏ: tạo một mảng `m*n` phần tử (vd `parent = [-1] * (m*n)`) là hoàn toàn thoải mái. |
| `k <= 10^4` | Nếu mỗi thao tác ta **duyệt lại cả lưới** (`N` ô) thì tốn `k · N = 10^8` bước — quá chậm với Python (dễ **TLE** — Time Limit Exceeded, quá thời gian). |
| `k` có thể **lớn hơn** `m * n` | Ví dụ `m = n = 1` mà `k = 10^4`: chỉ có 1 ô nên chắc chắn **có vị trí bị thêm lặp lại** (nguyên lý chuồng bồ câu). Test của LeetCode thật sự có ô trùng — code phải xử lý. |
| `0 <= ri < m`, `0 <= ci < n` | Mọi vị trí đều hợp lệ, không cần kiểm tra input. Nhưng khi xét **hàng xóm** của ô thì phải tự kiểm tra biên. |
| Follow-up `O(k log(mn))` | Gợi ý rằng mỗi thao tác chỉ nên tốn khoảng `log` — không được đụng tới cả lưới. Union-Find đạt (thậm chí tốt hơn) mức này. |

In [1]:
from typing import List
from collections import deque
import random
import time
import sys

# 4 hướng: lên, xuống, trái, phải (thứ tự này dùng xuyên suốt notebook)
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

# Hai ví dụ của đề
EX1 = (3, 3, [[0, 0], [0, 1], [1, 2], [2, 1]])          # đáp án [1, 1, 2, 3]
EX2 = (1, 1, [[0, 0]])                                  # đáp án [1]

# Ví dụ có ô bị thêm trùng (dạng test LeetCode hay có)
DUP = (3, 3, [[0, 0], [0, 1], [1, 2], [1, 2]])          # đáp án [1, 1, 2, 2]

# Ví dụ "khó" tự chọn để trace: có gộp 3 đảo một lúc, có ô trùng,
# có 2 hàng xóm thuộc CÙNG một đảo, có hai ô chỉ chạm chéo nhau
TRACE = (3, 4, [[0, 0], [0, 1], [2, 0], [1, 3], [2, 2], [0, 1], [1, 1], [2, 1], [1, 2]])
TRACE_ANSWER = [1, 1, 2, 3, 4, 4, 4, 2, 1]               # tính tay ở mục 2.2

for name, (m, n, pos) in [("EX1", EX1), ("EX2", EX2), ("DUP", DUP), ("TRACE", TRACE)]:
    print(f"{name:<6} m={m}, n={n}, k={len(pos)}, positions={pos}")

EX1    m=3, n=3, k=4, positions=[[0, 0], [0, 1], [1, 2], [2, 1]]
EX2    m=1, n=1, k=1, positions=[[0, 0]]
DUP    m=3, n=3, k=4, positions=[[0, 0], [0, 1], [1, 2], [1, 2]]
TRACE  m=3, n=4, k=9, positions=[[0, 0], [0, 1], [2, 0], [1, 3], [2, 2], [0, 1], [1, 1], [2, 1], [1, 2]]


## 2. Hiểu đề bằng tay trước khi code

### 2.1 Vẽ lưới sau mỗi thao tác của Example 1

Quy ước vẽ: `.` là nước, `#` là đất, `@` là ô **vừa** được thêm ở bước đó.

Điều quan trọng nhất cần thấy ở Example 1 là **bước #4**: ô `(2,1)` và ô `(1,2)` chỉ chạm nhau **theo đường chéo**. Đề chỉ nối các ô kề nhau **ngang hoặc dọc** (4 hướng), nên chạm chéo **không** làm chúng thành một đảo → có 3 đảo chứ không phải 2.

Bảng dưới là "lời giải bằng mắt" của Example 1:

| Bước | Ô thêm | Chạm vào đảo nào? | Số đảo |
|---:|---|---|---:|
| 1 | `(0,0)` | không chạm ai → đảo mới | 1 |
| 2 | `(0,1)` | chạm `(0,0)` bên trái → nhập vào đảo đó | 1 |
| 3 | `(1,2)` | xung quanh toàn nước → đảo mới | 2 |
| 4 | `(2,1)` | chỉ chạm chéo `(1,2)` → không tính → đảo mới | 3 |

Cell dưới viết hàm `draw_grid` (vẽ lưới thành các dòng chữ) và `show_ops` (in các lưới **cạnh nhau** sau từng bước để dễ so sánh). Các ô đất được giữ trong một `set` gồm các cặp `(r, c)` — `set` và `tuple` sẽ được giải thích kỹ ở mục 3.3; lúc này chỉ cần hiểu `(r, c) in lands` nghĩa là "ô `(r, c)` đã là đất chưa?".

In [2]:
def draw_grid(m, n, lands, new=None):
    # Trả về list các dòng chữ: '#' là đất, '.' là nước, '@' là ô vừa thêm
    rows = []
    for r in range(m):
        line = ""
        for c in range(n):
            if (r, c) == new:
                line += "@"
            elif (r, c) in lands:
                line += "#"
            else:
                line += "."
        rows.append(line)
    return rows


def show_ops(m, n, positions, answers, per_row=5):
    # In lưới sau từng thao tác, xếp cạnh nhau; answers là số đảo (để in kèm)
    lands = set()
    blocks = []
    for i, (r, c) in enumerate(positions):
        lands.add((r, c))
        title = f"#{i + 1} ({r},{c})"
        width = max(len(title), n, 7)
        body = draw_grid(m, n, lands, new=(r, c))
        footer = f"đảo={answers[i]}"
        blocks.append([title.ljust(width)] + [row.ljust(width) for row in body] + [footer.ljust(width)])
    for start in range(0, len(blocks), per_row):
        group = blocks[start:start + per_row]
        for line_idx in range(len(group[0])):
            print("   ".join(b[line_idx] for b in group))
        print()


m, n, pos = EX1
show_ops(m, n, pos, [1, 1, 2, 3])

#1 (0,0)   #2 (0,1)   #3 (1,2)   #4 (2,1)
@..        #@.        ##.        ##.     
...        ...        ..@        ..#     
...        ...        ...        .@.     
đảo=1      đảo=1      đảo=2      đảo=3   



### 2.2 Ô bị thêm trùng, và ví dụ "khó" dùng để trace

**Ô trùng.** Phát biểu đề nói "biến **nước** thành đất", nhưng các test của LeetCode có những vị trí xuất hiện **nhiều lần** trong `positions` (như đã thấy ở constraints: khi `k > m*n` thì chắc chắn phải trùng). Thêm đất lên một ô **đã là đất** thì lưới không đổi → số đảo **giữ nguyên**. Ví dụ `DUP`: `m = n = 3`, `positions = [[0,0],[0,1],[1,2],[1,2]]` → `[1, 1, 2, 2]`. Ta sẽ thấy ở mục 8 rằng nếu quên xử lý chuyện này, code Union-Find cho kết quả **sai**.

**Ví dụ `TRACE`** (`m = 3, n = 4`, 9 thao tác) được chọn để có đủ các tình huống "khó":

| Bước | Ô thêm | Chuyện gì xảy ra? | Số đảo |
|---:|---|---|---:|
| 1 | `(0,0)` | đảo mới (gọi là đảo A) | 1 |
| 2 | `(0,1)` | chạm A → nhập vào A | 1 |
| 3 | `(2,0)` | đảo mới B | 2 |
| 4 | `(1,3)` | đảo mới C | 3 |
| 5 | `(2,2)` | chỉ chạm **chéo** C → đảo mới D | 4 |
| 6 | `(0,1)` | **trùng** — đã là đất → không đổi | 4 |
| 7 | `(1,1)` | chạm A (ô `(0,1)` ở trên) → nhập vào A | 4 |
| 8 | `(2,1)` | chạm **3 đảo khác nhau** A (trên), B (trái), D (phải) → cả 3 cùng ô mới thành **1** đảo | 4 + 1 − 3 = 2 |
| 9 | `(1,2)` | chạm `(2,2)` và `(1,1)` — **cả hai cùng thuộc một đảo** (đã gộp ở bước 8) — và chạm C | 2 + 1 − 2 = 1 |

Bước 9 là "cái bẫy": ô mới có **3** hàng xóm là đất nhưng chỉ thuộc **2** đảo khác nhau. Ai đếm theo số hàng xóm sẽ ra `0` — sai.

In [3]:
m, n, pos = DUP
print("DUP:")
show_ops(m, n, pos, [1, 1, 2, 2])

m, n, pos = TRACE
print("TRACE:")
show_ops(m, n, pos, TRACE_ANSWER)

DUP:
#1 (0,0)   #2 (0,1)   #3 (1,2)   #4 (1,2)
@..        #@.        ##.        ##.     
...        ...        ..@        ..@     
...        ...        ...        ...     
đảo=1      đảo=1      đảo=2      đảo=2   

TRACE:
#1 (0,0)   #2 (0,1)   #3 (2,0)   #4 (1,3)   #5 (2,2)
@...       #@..       ##..       ##..       ##..    
....       ....       ....       ...@       ...#    
....       ....       @...       #...       #.@.    
đảo=1      đảo=1      đảo=2      đảo=3      đảo=4   

#6 (0,1)   #7 (1,1)   #8 (2,1)   #9 (1,2)
#@..       ##..       ##..       ##..    
...#       .@.#       .#.#       .#@#    
#.#.       #.#.       #@#.       ###.    
đảo=4      đảo=4      đảo=2      đảo=1   



## 3. Kiến thức nền

Bài này là bài **đồ thị trên lưới**. Trước khi giải, ta học lần lượt: lưới 2D trong Python → đánh số ô và tìm hàng xóm → `set`/`tuple`/`deque` → BFS loang và bài #200 → DFS bằng stack → gán nhãn đảo. Union-Find được dạy riêng ở mục 7, ngay trước khi dùng.

### 3.1 Lưới 2D = list lồng list

Lưới `m` hàng, `n` cột thường lưu thành **list của các list**: `grid[r]` là hàng thứ `r` (một list), `grid[r][c]` là ô ở hàng `r`, cột `c`. Hàng đánh số từ trên xuống `0..m-1`, cột từ trái sang `0..n-1`.

Có một cái bẫy kinh điển khi tạo lưới:

```python
grid = [[0] * n] * m            # SAI: m hàng là CÙNG MỘT list (bị "nhân bản tham chiếu")
grid = [[0] * n for _ in range(m)]   # ĐÚNG: mỗi hàng là một list riêng
```

Với cách sai, sửa `grid[0][0] = 1` sẽ làm **mọi hàng** đều đổi ở cột 0, vì thật ra chỉ có một list hàng duy nhất được tham chiếu `m` lần. Cell dưới cho bạn tận mắt thấy.

In [4]:
m, n = 3, 4

bad = [[0] * n] * m
bad[0][0] = 1
print("Cách SAI :", bad)          # cả 3 hàng đều có số 1 ở cột 0!

good = [[0] * n for _ in range(m)]
good[0][0] = 1
print("Cách ĐÚNG:", good)

# Đọc / ghi một ô: grid[r][c]
good[2][3] = 1
for row in good:
    print(row)
print("Số hàng =", len(good), "| số cột =", len(good[0]))

Cách SAI : [[1, 0, 0, 0], [1, 0, 0, 0], [1, 0, 0, 0]]
Cách ĐÚNG: [[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
[1, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 1]
Số hàng = 3 | số cột = 4


### 3.2 Đánh số ô `id = r * n + c` và 4 hàng xóm

**Đánh số phẳng.** Union-Find (mục 7) làm việc với các phần tử đánh số `0, 1, 2, ...`. Ta "trải phẳng" lưới theo từng hàng: ô `(r, c)` có số thứ tự

```text
id = r * n + c          (n là SỐ CỘT, không phải số hàng!)
```

Với `m = 3, n = 4`:

```text
         c=0  c=1  c=2  c=3
r=0:      0    1    2    3
r=1:      4    5    6    7
r=2:      8    9   10   11
```

Ví dụ ô `(2, 1)` → `2*4 + 1 = 9`. Chiều ngược lại dùng `divmod(id, n)` — trả về cặp (thương, dư): `divmod(9, 4) = (2, 1)`.

**4 hàng xóm.** Ô `(r, c)` có tối đa 4 hàng xóm: lên `(r-1, c)`, xuống `(r+1, c)`, trái `(r, c-1)`, phải `(r, c+1)`. Ta viết gọn bằng list hướng `DIRS = [(-1,0), (1,0), (0,-1), (0,1)]` và cộng từng cặp `(dr, dc)`.

**Kiểm tra biên — bắt buộc phải kiểm tra CẢ hàng LẪN cột:**

```python
if 0 <= nr < m and 0 <= nc < n:   # hàng xóm nằm trong lưới
```

Hai lý do khiến người mới hay "toang" ở đây:

1. Python **không báo lỗi** với chỉ số âm: `a[-1]` là phần tử **cuối** list. Nên `grid[-1][c]` lặng lẽ đọc **hàng cuối** thay vì báo "ra ngoài lưới".
2. Nếu chỉ kiểm tra `0 <= id < m*n` thì ô `(0, 3)` (cuối hàng 0) có "hàng xóm phải" `(0, 4)` → `id = 4` — chính là ô `(1, 0)` ở **đầu hàng 1**! Hai ô này không hề kề nhau.

In [5]:
m, n = 3, 4
for r in range(m):
    print(" ".join(f"{r * n + c:>2}" for c in range(n)))

print("(2,1) -> id", 2 * n + 1, "| id 9 ->", divmod(9, n))


def neighbors(r, c, m, n):
    # Các hàng xóm 4 hướng nằm TRONG lưới
    result = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            result.append((nr, nc))
    return result


print("Góc   (0,0):", neighbors(0, 0, m, n))     # chỉ 2 hàng xóm
print("Cạnh  (0,2):", neighbors(0, 2, m, n))     # 3 hàng xóm
print("Giữa  (1,1):", neighbors(1, 1, m, n))     # đủ 4

row0 = [10, 11, 12]
print("Chỉ số âm không báo lỗi: row0[-1] =", row0[-1])
print("Bẫy id: hàng xóm phải của (0,3) có id", 0 * n + 4, "=", divmod(4, n), "<- KHÔNG kề (0,3)!")

 0  1  2  3
 4  5  6  7
 8  9 10 11
(2,1) -> id 9 | id 9 -> (2, 1)
Góc   (0,0): [(1, 0), (0, 1)]
Cạnh  (0,2): [(1, 2), (0, 1), (0, 3)]
Giữa  (1,1): [(0, 1), (2, 1), (1, 0), (1, 2)]
Chỉ số âm không báo lỗi: row0[-1] = 12
Bẫy id: hàng xóm phải của (0,3) có id 4 = (1, 0) <- KHÔNG kề (0,3)!


### 3.3 `set` + `tuple` để lưu tập ô đất, `deque` làm hàng đợi

**`tuple` làm "tọa độ".** Cặp `(r, c)` là một `tuple` — giống list nhưng **không sửa được** (immutable). Nhờ không sửa được nên tuple dùng làm phần tử của `set` hoặc key của `dict` được (list thì không).

**`set`** là tập hợp không trùng lặp: `lands.add((0, 1))` thêm một ô; `(0, 1) in lands` kiểm tra "ô này là đất chưa?" trong thời gian **trung bình `O(1)`** (nhờ bảng băm — hash table), nhanh hơn nhiều so với tìm trong list (`O(độ dài list)`). Thêm một phần tử đã có vào set thì set **không đổi** — đúng tinh thần "thêm đất lên ô đã là đất".

**Hàng đợi (queue)** hoạt động kiểu **FIFO** (First In, First Out — vào trước ra trước), như xếp hàng mua vé. Trong Python dùng `collections.deque`:

- `q.append(x)`: cho `x` vào **cuối** hàng — `O(1)`.
- `q.popleft()`: lấy người ở **đầu** hàng ra — `O(1)`.

Đừng dùng `list.pop(0)` làm hàng đợi: mỗi lần pop đầu list, Python phải dời toàn bộ phần tử còn lại → `O(độ dài)`.

In [6]:
lands = set()
lands.add((0, 1))
lands.add((2, 3))
lands.add((0, 1))              # thêm trùng: set không đổi
print("lands =", lands, "| số ô đất =", len(lands))
print("(0,1) là đất?", (0, 1) in lands, "| (1,1) là đất?", (1, 1) in lands)

q = deque()
q.append("An")
q.append("Bình")
q.append("Chi")
print("Hàng đợi:", list(q))
print("popleft ->", q.popleft(), "| còn lại:", list(q))
print("popleft ->", q.popleft(), "| còn lại:", list(q))

lands = {(0, 1), (2, 3)} | số ô đất = 2
(0,1) là đất? True | (1,1) là đất? False
Hàng đợi: ['An', 'Bình', 'Chi']
popleft -> An | còn lại: ['Bình', 'Chi']
popleft -> Bình | còn lại: ['Chi']


### 3.4 BFS loang (flood fill) và ôn bài #200 Number of Islands

**Bài #200 (Medium)**: cho lưới `grid` gồm `"1"` (đất) và `"0"` (nước), đếm số đảo. Signature: `def numIslands(self, grid: List[List[str]]) -> int`. Đây chính là bài #305 nhưng **chỉ hỏi một lần** cho một lưới cố định.

**Ý tưởng "loang":** tưởng tượng ta đổ sơn vào một ô đất — sơn lan sang mọi ô đất kề nó, rồi kề của kề... cho tới khi phủ kín **cả hòn đảo**. Mỗi lần phải đổ sơn mới (gặp một ô đất **chưa bị sơn**) là ta vừa phát hiện **một đảo mới**.

Thuật toán:

```text
count = 0, visited = rỗng
duyệt mọi ô (r, c) theo thứ tự:
    nếu là đất và chưa thăm:
        count += 1                       # phát hiện đảo mới
        BFS từ (r, c): đánh dấu mọi ô đất nối được là đã thăm
return count
```

**BFS** (Breadth-First Search — tìm kiếm theo chiều rộng) dùng hàng đợi: lấy một ô ra, xem 4 hàng xóm, hàng xóm nào là đất và **chưa thăm** thì **đánh dấu ngay** và cho vào hàng đợi. Phải đánh dấu **lúc đưa vào hàng đợi** (không phải lúc lấy ra) để một ô không bị đưa vào nhiều lần.

Độ phức tạp: mỗi ô được đưa vào hàng đợi tối đa 1 lần, mỗi lần xét 4 hàng xóm → `O(m·n)` thời gian, `O(m·n)` bộ nhớ cho `visited`.

In [7]:
def num_islands_bfs(grid):
    # grid: list lồng list, ô đất là "1" hoặc 1 (chấp nhận cả hai kiểu)
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] in ("1", 1) and not visited[r][c]:
                count += 1                          # phát hiện đảo mới
                visited[r][c] = True
                q = deque([(r, c)])
                while q:
                    x, y = q.popleft()
                    for dr, dc in DIRS:
                        nx, ny = x + dr, y + dc
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] in ("1", 1) and not visited[nx][ny]:
                            visited[nx][ny] = True  # đánh dấu LÚC ĐƯA VÀO
                            q.append((nx, ny))
    return count


# Hai ví dụ của bài #200
g200_1 = [["1", "1", "1", "1", "0"],
          ["1", "1", "0", "1", "0"],
          ["1", "1", "0", "0", "0"],
          ["0", "0", "0", "0", "0"]]
g200_2 = [["1", "1", "0", "0", "0"],
          ["1", "1", "0", "0", "0"],
          ["0", "0", "1", "0", "0"],
          ["0", "0", "0", "1", "1"]]
print("#200 Example 1:", num_islands_bfs(g200_1))   # 1
print("#200 Example 2:", num_islands_bfs(g200_2))   # 3
assert num_islands_bfs(g200_1) == 1 and num_islands_bfs(g200_2) == 3

#200 Example 1: 1
#200 Example 2: 3


### 3.5 DFS bằng stack — cùng ý "loang", khác thứ tự đi

**DFS** (Depth-First Search — tìm theo chiều sâu) cũng loang hết một đảo, nhưng thay hàng đợi bằng **ngăn xếp** (stack, **LIFO** — Last In, First Out: vào sau ra trước, như chồng đĩa). Trong Python, list làm stack rất tốt: `st.append(x)` đẩy vào đỉnh, `st.pop()` lấy ở đỉnh, cả hai `O(1)`.

Để đếm đảo, **thứ tự** thăm không quan trọng — BFS hay DFS đều thăm đúng tập ô của đảo. Vì vậy hai hàm phải cho cùng kết quả.

Vì sao không viết DFS **đệ quy** (hàm tự gọi lại chính nó)? Viết đệ quy rất gọn, nhưng Python giới hạn độ sâu đệ quy khoảng **1000** tầng (`sys.getrecursionlimit()`). Một hòn đảo hình "con rắn" 10 000 ô sẽ làm đệ quy sâu 10 000 tầng → `RecursionError`. Dùng stack tự quản lý thì không bị giới hạn này. Ta sẽ gặp lại đúng vấn đề này với `find` đệ quy ở mục 13.

In [8]:
def num_islands_dfs(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] in ("1", 1) and not visited[r][c]:
                count += 1
                visited[r][c] = True
                st = [(r, c)]                       # stack thay cho queue
                while st:
                    x, y = st.pop()                 # lấy ở ĐỈNH stack
                    for dr, dc in DIRS:
                        nx, ny = x + dr, y + dc
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] in ("1", 1) and not visited[nx][ny]:
                            visited[nx][ny] = True
                            st.append((nx, ny))
    return count


print("DFS #200 Example 1:", num_islands_dfs(g200_1))
print("DFS #200 Example 2:", num_islands_dfs(g200_2))
print("Giới hạn đệ quy mặc định của Python:", sys.getrecursionlimit())
assert num_islands_dfs(g200_1) == num_islands_bfs(g200_1)
assert num_islands_dfs(g200_2) == num_islands_bfs(g200_2)

DFS #200 Example 1: 1
DFS #200 Example 2: 3
Giới hạn đệ quy mặc định của Python: 1000


### 3.6 Không chỉ đếm: **gán nhãn** cho từng đảo

Nếu khi loang ta ghi **số thứ tự đảo** vào từng ô (đảo thứ nhất ghi `0`, đảo thứ hai ghi `1`, ...), ta có một **lưới nhãn** (label grid). Lưới nhãn trả lời ngay câu hỏi quan trọng nhất của bài này:

> **Hai ô đất có thuộc cùng một đảo không?** → So nhãn của chúng.

Ví dụ #200 Example 2 có 3 đảo, nhãn sẽ là `0`, `1`, `2`. Ý tưởng "nhãn" chính là nền của **Cách 2**, và Union-Find (Cách 3) là phiên bản "nhãn thông minh" của nó.

In [9]:
def label_islands(grid):
    # Trả về (lưới nhãn, số đảo); nước có nhãn -1
    m, n = len(grid), len(grid[0])
    label = [[-1] * n for _ in range(m)]
    next_label = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] in ("1", 1) and label[r][c] == -1:
                label[r][c] = next_label
                q = deque([(r, c)])
                while q:
                    x, y = q.popleft()
                    for dr, dc in DIRS:
                        nx, ny = x + dr, y + dc
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] in ("1", 1) and label[nx][ny] == -1:
                            label[nx][ny] = next_label
                            q.append((nx, ny))
                next_label += 1
    return label, next_label


def show_labels(label):
    for row in label:
        print(" ".join("." if v == -1 else str(v) for v in row))


lab, cnt = label_islands(g200_2)
show_labels(lab)
print("Số đảo =", cnt)

0 0 . . .
0 0 . . .
. . 1 . .
. . . 2 2
Số đảo = 3


## 4. Cách 1 — Sau mỗi lần thêm, đếm lại toàn bộ đảo (trọng tài)

Đây là cách người mới nghĩ ra đầu tiên, và nó **đúng hiển nhiên**: ta đã có hàm đếm đảo của bài #200, vậy cứ mỗi thao tác thì

1. biến ô `(r, c)` thành đất trong lưới,
2. gọi lại `num_islands_bfs(grid)` để đếm từ đầu,
3. ghi kết quả vào `answer`.

Ô trùng **tự động** được xử lý đúng: gán `grid[r][c] = 1` lần thứ hai không làm lưới thay đổi, nên số đảo giữ nguyên.

Vì quá dễ kiểm chứng, Cách 1 sẽ đóng vai **trọng tài** (reference / oracle): mọi cách nhanh hơn ở sau đều phải cho **cùng đáp án** với nó trên hàng trăm input ngẫu nhiên.

**Độ phức tạp:** mỗi lần đếm tốn `O(m·n)` (duyệt mọi ô + BFS), làm `k` lần → **`O(k · m · n)`** thời gian, `O(m·n)` bộ nhớ.

In [10]:
def num_islands2_recount(m, n, positions):
    grid = [[0] * n for _ in range(m)]
    answer = []
    for r, c in positions:
        grid[r][c] = 1                       # ô trùng: gán lại 1 cũng không sao
        answer.append(num_islands_bfs(grid)) # đếm lại TỪ ĐẦU cả lưới
    return answer


print("EX1  :", num_islands2_recount(*EX1))
print("EX2  :", num_islands2_recount(*EX2))
print("DUP  :", num_islands2_recount(*DUP))
print("TRACE:", num_islands2_recount(*TRACE))
assert num_islands2_recount(*EX1) == [1, 1, 2, 3]
assert num_islands2_recount(*EX2) == [1]
assert num_islands2_recount(*DUP) == [1, 1, 2, 2]
assert num_islands2_recount(*TRACE) == TRACE_ANSWER

EX1  : [1, 1, 2, 3]
EX2  : [1]
DUP  : [1, 1, 2, 2]
TRACE: [1, 1, 2, 3, 4, 4, 4, 2, 1]


### Nút thắt của Cách 1: mỗi thao tác chỉ đổi **1 ô**, nhưng ta nhìn lại **cả lưới**

Với giới hạn đề `k = 10^4` và `m·n = 10^4`, tổng công việc là `k · m · n = 10^8` lượt "nhìn một ô" — mỗi lượt trong Python lại gồm vài phép so sánh, kiểm tra biên... Ta đo trên vài lưới vuông nhỏ (lấy `k = m·n` vị trí ngẫu nhiên) rồi **ngoại suy** (ước lượng theo tỉ lệ) cho kích thước thật.

Khi đọc kết quả, để ý: lưới cạnh gấp đôi thì `N` gấp 4 và `k` gấp 4 → thời gian gấp khoảng **16** lần. Đó là dấu hiệu của tăng trưởng `k · N`.

In [11]:
def random_positions(m, n, k, seed):
    rng = random.Random(seed)
    return [[rng.randrange(m), rng.randrange(n)] for _ in range(k)]


timings = {}
for side in [10, 20, 40]:
    k = side * side
    pos = random_positions(side, side, k, seed=305)
    t0 = time.perf_counter()
    num_islands2_recount(side, side, pos)
    timings[side] = time.perf_counter() - t0
    print(f"lưới {side:>2}x{side:<2} k={k:>4}: k*N = {k * side * side:>9,} lượt nhìn ô, thời gian {timings[side]:.3f}s")

scale = (10**4 * 10**4) / (1600 * 1600)
print(f"Ước lượng cho m*n = 10^4, k = 10^4: khoảng {timings[40] * scale:.0f} giây  -> chắc chắn TLE")

lưới 10x10 k= 100: k*N =    10,000 lượt nhìn ô, thời gian 0.003s
lưới 20x20 k= 400: k*N =   160,000 lượt nhìn ô, thời gian 0.046s


lưới 40x40 k=1600: k*N = 2,560,000 lượt nhìn ô, thời gian 0.675s
Ước lượng cho m*n = 10^4, k = 10^4: khoảng 26 giây  -> chắc chắn TLE


## 5. Bước đột phá: thêm một ô chỉ thay đổi **cục bộ**

Hãy nghĩ xem khi ô `x` từ nước thành đất, các đảo thay đổi thế nào:

- Đảo nào **không chạm** `x` thì **y nguyên** — chẳng có lý do gì để nó đổi.
- Các đảo **chạm** `x` (có ô đất là một trong 4 hàng xóm của `x`) giờ được `x` "bắc cầu" nối lại → chúng **cùng `x`** trở thành **một** đảo duy nhất.

Vì sao chỉ cần xét 4 hàng xóm? Mọi con đường mới (nối hai đảo cũ) đều **phải đi qua `x`** — vì trước khi có `x`, hai đảo đó không nối với nhau. Mà đi qua `x` thì phải bước vào `x` từ một trong 4 ô kề nó.

Gọi `t` là **số đảo khác nhau** chạm vào `x`. Ta mất `t` đảo cũ và được `1` đảo mới:

```text
số đảo mới = số đảo cũ + 1 - t
```

| Tình huống | `t` | Thay đổi |
|---|---:|---:|
| Xung quanh toàn nước | 0 | `+1` (đảo mới) |
| Chạm 1 đảo (dù chạm bằng 1, 2 hay 3 ô của đảo đó) | 1 | `0` |
| Chạm 2 đảo khác nhau | 2 | `−1` |
| Chạm 4 đảo khác nhau (ô ở giữa hình chữ thập) | 4 | `−3` |

Chữ quan trọng nhất là **"khác nhau"**: ở bước 9 của `TRACE`, ô mới có **3** hàng xóm là đất nhưng chỉ **2** đảo khác nhau → `t = 2`, không phải 3.

Vậy toàn bộ bài toán quy về **một câu hỏi con** được hỏi đi hỏi lại:

> **Hai ô đất này có thuộc cùng một đảo không?** — và khi chúng được nối, **gộp** hai đảo thành một.

Cell dưới kiểm chứng công thức trên vài lưới nhỏ: dùng `label_islands` để biết nhãn đảo của các hàng xóm, dự đoán bằng công thức, rồi so với đếm lại thật.

In [12]:
def delta_if_add(grid, r, c):
    m, n = len(grid), len(grid[0])
    label, before = label_islands(grid)
    touched = set()                              # nhãn các đảo KHÁC NHAU chạm vào (r, c)
    for nr, nc in neighbors(r, c, m, n):
        if label[nr][nc] != -1:
            touched.add(label[nr][nc])
    predicted = before + 1 - len(touched)
    after_grid = [row[:] for row in grid]        # bản sao để thử thêm đất
    after_grid[r][c] = 1
    real = num_islands_bfs(after_grid)
    return before, touched, predicted, real


G_PLUS  = [[0, 1, 0],
           [1, 0, 1],
           [0, 1, 0]]       # 4 đảo một ô bao quanh ô giữa
G_HOOK  = [[0, 1, 1],
           [1, 0, 1],
           [0, 0, 0]]       # đảo hình móc câu + một đảo lẻ
G_FAR   = [[1, 0, 0],
           [0, 0, 0],
           [0, 0, 1]]       # ô giữa không chạm ai

for name, g, cell in [("G_PLUS", G_PLUS, (1, 1)), ("G_HOOK", G_HOOK, (1, 1)), ("G_FAR", G_FAR, (1, 1))]:
    before, touched, predicted, real = delta_if_add(g, *cell)
    print(f"{name:<7} thêm {cell}: đảo trước={before}, nhãn chạm={sorted(touched)}, "
          f"dự đoán {before}+1-{len(touched)}={predicted}, đếm lại thật={real}")
    assert predicted == real

G_PLUS  thêm (1, 1): đảo trước=4, nhãn chạm=[0, 1, 2, 3], dự đoán 4+1-4=1, đếm lại thật=1
G_HOOK  thêm (1, 1): đảo trước=2, nhãn chạm=[0, 1], dự đoán 2+1-2=1, đếm lại thật=1
G_FAR   thêm (1, 1): đảo trước=2, nhãn chạm=[], dự đoán 2+1-0=3, đếm lại thật=3


## 6. Cách 2 — Duy trì lưới nhãn, khi gộp thì đổi nhãn đảo nhỏ (trung gian)

Cách 1 tính lại lưới nhãn **từ đầu** mỗi bước. Cách 2 giữ lưới nhãn **sống** qua các bước và chỉ sửa phần cần sửa.

Dữ liệu cần giữ:

- `label[r][c]`: nhãn đảo của ô, `-1` nếu là nước.
- `size[L]`: số ô của đảo có nhãn `L` (dùng `dict`).
- `count`: số đảo hiện tại.

Khi thêm ô `(r, c)`:

1. Nếu `label[r][c] != -1` → **ô trùng**, ghi lại `count` cũ và bỏ qua.
2. Gom **nhãn khác nhau** của các hàng xóm là đất vào một `dict` `nb` (nhãn → một ô thuộc đảo đó). Dùng `dict`/`set` nên hai hàng xóm cùng nhãn chỉ tính **một lần**.
3. Nếu `nb` rỗng → đảo mới với nhãn mới.
4. Ngược lại, chọn đảo **lớn nhất** trong `nb` làm đảo "giữ nhãn" (`keep`). Ô mới nhận nhãn `keep`. Mỗi đảo còn lại (nhỏ hơn hoặc bằng) bị **đổi nhãn** thành `keep` bằng một lượt **BFS chỉ đi trên các ô mang nhãn cũ**.
5. `count += 1 - len(nb)` — đúng công thức ở mục 5.

Vì sao phải chọn **đảo lớn** làm `keep`? Vì chi phí đổi nhãn bằng **số ô bị đổi**, nên ta luôn đổi phần **nhỏ**. Mục kế tiếp chứng minh mẹo này cứu ta khỏi `O(N²)`.

Hàm dưới có thêm tham số `keep_rule` (để thí nghiệm chọn sai), `stats` (đếm tổng số ô bị đổi nhãn) và `verbose` (in trace).

In [13]:
def num_islands2_relabel(m, n, positions, keep_rule="big", stats=None, verbose=False):
    label = [[-1] * n for _ in range(m)]
    size = {}                      # nhãn -> số ô
    next_label = 0
    count = 0
    answer = []
    relabeled = 0
    for r, c in positions:
        if label[r][c] != -1:                       # 1. ô trùng
            answer.append(count)
            if verbose:
                print(f"({r},{c}) trùng -> giữ count={count}")
            continue
        nb = {}                                     # 2. nhãn khác nhau -> một ô của đảo đó
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and label[nr][nc] != -1:
                if label[nr][nc] not in nb:
                    nb[label[nr][nc]] = (nr, nc)
        if not nb:                                  # 3. đảo mới
            label[r][c] = next_label
            size[next_label] = 1
            next_label += 1
            note = "đảo mới"
        else:                                       # 4. gộp vào đảo keep
            if keep_rule == "big":
                keep = max(nb, key=lambda L: size[L])
            else:                                   # chọn SAI: giữ đảo nhỏ nhất
                keep = min(nb, key=lambda L: size[L])
            label[r][c] = keep
            size[keep] += 1
            moved = 0
            for old, (sr, sc) in nb.items():
                if old == keep:
                    continue
                label[sr][sc] = keep                # BFS đổi nhãn cả đảo 'old'
                moved += 1
                q = deque([(sr, sc)])
                while q:
                    x, y = q.popleft()
                    for dr, dc in DIRS:
                        a, b = x + dr, y + dc
                        if 0 <= a < m and 0 <= b < n and label[a][b] == old:
                            label[a][b] = keep
                            moved += 1
                            q.append((a, b))
                size[keep] += size.pop(old)
            relabeled += moved
            note = f"nhãn chạm {list(nb)} -> giữ {keep}, đổi {moved} ô"
        count += 1 - len(nb)                        # 5. công thức cục bộ
        answer.append(count)
        if verbose:
            g = " / ".join("".join("." if v == -1 else str(v) for v in row) for row in label)
            print(f"({r},{c}) {note:<32} | {g} | count={count}")
    if stats is not None:
        stats["relabeled"] = relabeled
    return answer


for name, ex, expected in [("EX1", EX1, [1, 1, 2, 3]), ("EX2", EX2, [1]), ("DUP", DUP, [1, 1, 2, 2]), ("TRACE", TRACE, TRACE_ANSWER)]:
    got = num_islands2_relabel(*ex)
    print(f"{name:<6} {got}")
    assert got == expected

EX1    [1, 1, 2, 3]
EX2    [1]
DUP    [1, 1, 2, 2]
TRACE  [1, 1, 2, 3, 4, 4, 4, 2, 1]


### Trace Cách 2 với ví dụ `TRACE`

Lưới nhãn được in trên một dòng, các hàng cách nhau bởi ` / ` (hàng 0 / hàng 1 / hàng 2). Bảng dự kiến:

| Bước | Ô | Nhãn các đảo chạm vào | Hành động | Lưới nhãn sau bước | `count` |
|---:|---|---|---|---|---:|
| 1 | `(0,0)` | — | đảo mới nhãn `0` | `0... / .... / ....` | 1 |
| 2 | `(0,1)` | `{0}` | giữ `0`, không đổi ai | `00.. / .... / ....` | 1 |
| 3 | `(2,0)` | — | đảo mới nhãn `1` | `00.. / .... / 1...` | 2 |
| 4 | `(1,3)` | — | đảo mới nhãn `2` | `00.. / ...2 / 1...` | 3 |
| 5 | `(2,2)` | — (chỉ chéo với `(1,3)`) | đảo mới nhãn `3` | `00.. / ...2 / 1.3.` | 4 |
| 6 | `(0,1)` | trùng | bỏ qua | không đổi | 4 |
| 7 | `(1,1)` | `{0}` | giữ `0` | `00.. / .0.2 / 1.3.` | 4 |
| 8 | `(2,1)` | `{0, 1, 3}` | đảo `0` lớn nhất (3 ô) → giữ `0`; đổi nhãn đảo `1` và `3` (**2 ô**) | `00.. / .0.2 / 000.` | 4+1−3 = 2 |
| 9 | `(1,2)` | `{0, 2}` (hai hàng xóm `(2,2)`, `(1,1)` cùng nhãn `0` → tính 1 lần) | giữ `0`, đổi 1 ô | `00.. / .000 / 000.` | 2+1−2 = 1 |

Chạy để đối chiếu:

In [14]:
m, n, pos = TRACE
result = num_islands2_relabel(m, n, pos, verbose=True)
print("answer =", result)

(0,0) đảo mới                          | 0... / .... / .... | count=1
(0,1) nhãn chạm [0] -> giữ 0, đổi 0 ô  | 00.. / .... / .... | count=1
(2,0) đảo mới                          | 00.. / .... / 1... | count=2
(1,3) đảo mới                          | 00.. / ...2 / 1... | count=3
(2,2) đảo mới                          | 00.. / ...2 / 1.3. | count=4
(0,1) trùng -> giữ count=4
(1,1) nhãn chạm [0] -> giữ 0, đổi 0 ô  | 00.. / .0.2 / 1.3. | count=4
(2,1) nhãn chạm [0, 1, 3] -> giữ 0, đổi 2 ô | 00.. / .0.2 / 000. | count=2
(1,2) nhãn chạm [0, 2] -> giữ 0, đổi 1 ô | 00.. / .000 / 000. | count=1
answer = [1, 1, 2, 3, 4, 4, 4, 2, 1]


### Vì sao "gộp nhỏ vào lớn" lại quan trọng đến vậy? Độ phức tạp Cách 2

**Lập luận nhân đôi (doubling argument).** Xét một ô `x` bất kỳ. Mỗi khi `x` bị đổi nhãn, đảo của nó (kích thước `s`) đang được nhập vào một đảo `keep` có kích thước **`≥ s`** (vì ta luôn giữ đảo lớn nhất). Sau khi gộp, đảo chứa `x` có kích thước **`≥ 2s`** — tức là **ít nhất gấp đôi**. Kích thước không thể vượt `N = m·n`, nên một ô bị đổi nhãn **tối đa `log₂ N`** lần (với `N = 10^4` thì `log₂ N ≈ 13.3`, tức tối đa 13 lần).

→ Tổng số lần đổi nhãn trên toàn bộ thuật toán `≤ N · log₂ N`. Cộng với `O(1)` việc cho mỗi thao tác (xét 4 hàng xóm):

- **Thời gian: `O(k + N log N)`**, với `N = m·n`.
- **Bộ nhớ: `O(N)`** cho lưới nhãn (+ hàng đợi BFS).

**Nếu chọn sai** (giữ nhãn của đảo **nhỏ**, đổi nhãn đảo **lớn**) thì lập luận nhân đôi sụp đổ. Input "cái lược" dưới đây là phản ví dụ: hàng 0 là một đảo dài `W` ô; sau đó lần lượt thêm một đảo 1 ô ở hàng 2 rồi một ô "cầu" ở hàng 1 nối nó với đảo lớn. Mỗi lần nối, cách chọn sai phải đổi nhãn **cả đảo lớn** (kích thước `W`, `W+2`, `W+4`, ...) → tổng `≈ 3W²/4` (với `W = 100`: `7450`); cách đúng chỉ đổi đúng 1 ô mỗi lần → tổng `= W/2`.

In [15]:
def comb_positions(W):
    pos = [[0, c] for c in range(W)]       # hàng 0: một đảo dài W ô
    for c in range(0, W, 2):
        pos.append([2, c])                 # đảo 1 ô ở hàng 2
        pos.append([1, c])                 # ô cầu: chạm đảo lớn (trên) và đảo 1 ô (dưới)
    return pos


pos6 = comb_positions(6)
for row in draw_grid(3, 6, {tuple(p) for p in pos6}):
    print("   ", row)

for W in [100, 200, 400]:
    good, bad = {}, {}
    a1 = num_islands2_relabel(3, W, comb_positions(W), "big", good)
    a2 = num_islands2_relabel(3, W, comb_positions(W), "small", bad)
    assert a1 == a2
    print(f"W={W:>3}: gộp nhỏ vào lớn đổi {good['relabeled']:>4} ô | chọn sai đổi {bad['relabeled']:>6} ô")

    ######
    #.#.#.
    #.#.#.
W=100: gộp nhỏ vào lớn đổi   50 ô | chọn sai đổi   7450 ô
W=200: gộp nhỏ vào lớn đổi  100 ô | chọn sai đổi  29900 ô
W=400: gộp nhỏ vào lớn đổi  200 ô | chọn sai đổi 119800 ô


## 7. Union-Find (DSU) từ con số 0

Cách 2 đã nhanh, nhưng mỗi lần gộp vẫn phải **viết lại nhãn của từng ô** trong đảo nhỏ, và code BFS đổi nhãn khá dài. **Union-Find** (còn gọi **DSU** — Disjoint Set Union, "hợp các tập rời nhau") làm việc gộp **lười hơn**: gộp hai nhóm chỉ cần **sửa đúng một con trỏ**.

### 7.1 Ý tưởng: ai cũng có một "người để hỏi"

Hình dung một lớp học chia thành nhiều nhóm. Mỗi nhóm có một **trưởng nhóm** (gọi là **gốc** — root). Mỗi bạn ghi trên tay tên **một người** để hỏi, gọi là `parent` (cha). Trưởng nhóm thì ghi tên **chính mình**.

- `parent[x]`: người mà `x` hỏi. Nếu `parent[x] == x` thì `x` là gốc.
- **`find(x)`**: muốn biết trưởng nhóm của `x`, ta hỏi `parent[x]`, rồi hỏi tiếp `parent` của người đó... cho tới khi gặp người tự ghi tên mình. Đó là gốc.
- **Hai người cùng nhóm ⇔ `find` của họ cho ra cùng một gốc.**
- **`union(a, b)`**: tìm gốc `ra = find(a)`, `rb = find(b)`. Nếu `ra == rb` thì đã cùng nhóm, trả về `False`. Ngược lại cho gốc `rb` ghi tên `ra` (`parent[rb] = ra`) — cả nhóm của `b` tự động thuộc nhóm của `a` — và trả về `True`.

Mỗi nhóm vì thế là một **cây** (tree) mà gốc ở trên cùng. Ví dụ `parent = [0, 0, 0, 2, 2, 5]` là "rừng" (forest — nhiều cây) sau:

```text
     0          5
    / \
   1   2
      / \
     3   4
```

`find(4)`: `4 → 2 → 0`, gặp `parent[0] == 0` → gốc là `0`. `find(5) = 5` nên `4` và `5` **khác** nhóm.

Điểm "ăn tiền" so với Cách 2: khi gộp nhóm `{2,3,4}` vào nhóm `{0,1}`, ta **không** sửa `parent` của 3 và 4, chỉ sửa `parent[2] = 0`. Cell dưới dựng đúng rừng trên bằng một chuỗi `union` và in `parent` sau mỗi bước. Để ý `union(0, 3)` trả về `False` vì hai người đã cùng nhóm — đây chính là tình huống "hàng xóm **cùng đảo**" ở bước 9 của `TRACE`.

In [16]:
parent = list(range(6))          # ban đầu mỗi người là một nhóm, tự làm gốc


def find_naive(x):
    while parent[x] != x:        # chưa phải gốc -> đi lên cha
        x = parent[x]
    return x


def union_naive(a, b):
    ra, rb = find_naive(a), find_naive(b)
    if ra == rb:
        return False             # đã cùng nhóm
    parent[rb] = ra              # gắn GỐC của b vào GỐC của a
    return True


print("ban đầu        parent =", parent)
groups = 6
for a, b in [(0, 1), (2, 3), (3, 4), (1, 4), (0, 3)]:
    ok = union_naive(a, b)
    groups -= ok                 # True được tính là 1, False là 0
    print(f"union({a},{b}) -> {str(ok):<5} parent = {parent} | số nhóm = {groups}")

print("find(4) =", find_naive(4), "| find(5) =", find_naive(5))
print("4 và 1 cùng nhóm?", find_naive(4) == find_naive(1), "| 4 và 5 cùng nhóm?", find_naive(4) == find_naive(5))

ban đầu        parent = [0, 1, 2, 3, 4, 5]
union(0,1) -> True  parent = [0, 0, 2, 3, 4, 5] | số nhóm = 5
union(2,3) -> True  parent = [0, 0, 2, 2, 4, 5] | số nhóm = 4
union(3,4) -> True  parent = [0, 0, 2, 2, 2, 5] | số nhóm = 3
union(1,4) -> True  parent = [0, 0, 0, 2, 2, 5] | số nhóm = 2
union(0,3) -> False parent = [0, 0, 0, 2, 2, 5] | số nhóm = 2
find(4) = 0 | find(5) = 5
4 và 1 cùng nhóm? True | 4 và 5 cùng nhóm? False


Hai nhận xét sẽ dùng thẳng cho bài toán:

1. **Mỗi `union` thành công làm số nhóm giảm đúng 1** (hai nhóm khác nhau nhập làm một). `union` thất bại thì số nhóm không đổi. Ở trên: 6 nhóm, 4 lần thành công → còn 2 nhóm.
2. Mỗi phép `union` chỉ sửa **một** ô của `parent`. Chi phí thật nằm ở `find` — đi từ một phần tử lên gốc. Vậy câu hỏi quyết định là: **cây cao bao nhiêu?**

### 7.2 Vấn đề: cây có thể thành "dây xích"

Với `union_naive`, ai làm gốc là do **thứ tự tham số** quyết định, không có kiểm soát. Thử chuỗi rất tự nhiên: mỗi phần tử mới `i+1` được nối với phần tử trước `i` bằng `union(i+1, i)`. Gốc của `i+1` là chính nó (đang lẻ loi) nên nó trở thành **gốc mới**, cả cây cũ bị treo bên dưới:

```text
union(1,0): 1           union(2,1): 2           ...   7
            |                       |                 |
            0                       1                 6
                                    |                 :
                                    0                 0      <- độ cao 7
```

**Độ cao** (height) của cây = số bước nhảy nhiều nhất từ một phần tử lên gốc. Với `n` phần tử ta có độ cao `n − 1`, và `find(0)` phải nhảy `n − 1` bước. Lặp lại nhiều lần thì DSU ngây thơ chậm **không kém** Cách 1.

In [17]:
def tree_height(parent):
    # Độ cao = số bước nhảy nhiều nhất từ một phần tử lên tới gốc
    best = 0
    for v in range(len(parent)):
        steps = 0
        while parent[v] != v:
            v = parent[v]
            steps += 1
        best = max(best, steps)
    return best


def find_count_hops(parent, x):
    hops = 0
    while parent[x] != x:
        x = parent[x]
        hops += 1
    return x, hops


parent = list(range(8))
for i in range(7):
    union_naive(i + 1, i)          # dùng lại union_naive (nó đọc biến toàn cục parent)
print("parent  =", parent)
print("độ cao  =", tree_height(parent))
root, hops = find_count_hops(parent, 0)
print("find(0) -> gốc", root, "sau", hops, "bước nhảy")

parent  = [1, 2, 3, 4, 5, 6, 7, 7]
độ cao  = 7
find(0) -> gốc 7 sau 7 bước nhảy


### 7.3 Tối ưu 1 — Path compression (nén đường đi)

**Ý tưởng:** đã mất công đi từ `x` lên tới gốc, thì nhân tiện **cho mọi phần tử trên đường đi trỏ thẳng vào gốc**. Lần sau hỏi lại bất kỳ ai trong số đó chỉ tốn 1 bước.

Viết theo **hai lượt** (dễ hiểu, không đệ quy):

```python
def find(x):
    root = x
    while parent[root] != root:     # lượt 1: tìm gốc
        root = parent[root]
    while parent[x] != root:        # lượt 2: cho từng người trên đường trỏ thẳng vào gốc
        nxt = parent[x]
        parent[x] = root
        x = nxt
    return root
```

Với dây xích ở trên, **một** lần `find(0)` tốn 7 bước, nhưng sau đó cả 7 phần tử đều trỏ thẳng vào gốc `7` — cây từ độ cao 7 thành **độ cao 1**:

```text
trước:  0 -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7        sau:        7
                                                       / / / | \ \ \
                                                      0 1 2  3  4 5 6
```

Hai cách viết khác hay gặp (cùng ý tưởng):

- Đệ quy một dòng: `if parent[x] != x: parent[x] = find(parent[x])` — gọn nhưng có thể chạm giới hạn đệ quy của Python khi cây còn cao.
- **Path halving** (chia đôi đường): `while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]` — mỗi bước cho `x` trỏ lên "ông" thay vì "cha", một lượt duy nhất.

In [18]:
def find_compress(parent, x):
    root = x
    while parent[root] != root:       # lượt 1: tìm gốc
        root = parent[root]
    while parent[x] != root:          # lượt 2: nén đường
        nxt = parent[x]
        parent[x] = root
        x = nxt
    return root


chain = [1, 2, 3, 4, 5, 6, 7, 7]      # dây xích 0 -> 1 -> ... -> 7 như ở trên
print("trước: parent =", chain, "| độ cao =", tree_height(chain))
print("find(0) tốn", find_count_hops(chain, 0)[1], "bước, gốc =", find_compress(chain, 0))
print("sau  : parent =", chain, "| độ cao =", tree_height(chain))
print("find(3) bây giờ tốn", find_count_hops(chain, 3)[1], "bước")

trước: parent = [1, 2, 3, 4, 5, 6, 7, 7] | độ cao = 7
find(0) tốn 7 bước, gốc = 7
sau  : parent = [7, 7, 7, 7, 7, 7, 7, 7] | độ cao = 1
find(3) bây giờ tốn 1 bước


### 7.4 Tối ưu 2 — Union by size / union by rank (gộp theo kích thước / hạng)

Path compression sửa cây **sau khi** nó đã cao. Tối ưu thứ hai **ngăn** cây cao ngay từ lúc gộp: **luôn treo cây nhỏ dưới gốc của cây lớn** (giống hệt "gộp nhỏ vào lớn" ở Cách 2!).

- **Union by size**: giữ `size[root]` = số phần tử của cây. Khi gộp, gốc có `size` nhỏ hơn trở thành con của gốc có `size` lớn hơn; cộng `size` vào gốc mới.
- **Union by rank**: giữ `rank[root]` = một **cận trên của độ cao** cây. Treo cây rank thấp dưới cây rank cao; nếu bằng nhau thì treo tùy ý và **tăng rank gốc mới thêm 1**.

**Vì sao union by size giữ độ cao `≤ log₂ n`?** Độ sâu của một phần tử `x` chỉ tăng (thêm 1) khi cây chứa `x` bị treo dưới một cây **lớn hơn hoặc bằng** nó → cây chứa `x` **ít nhất gấp đôi**. Kích thước tối đa là `n`, nên độ sâu của `x` tăng tối đa `log₂ n` lần. Với `n = 10^4`, cây cao không quá **13** — vẫn đúng lập luận nhân đôi của mục 6.

Cell dưới đóng gói thành class `DSU` (union by size + path compression) — đây là "bộ đồ nghề" chuẩn. Kèm theo `DSURank` để bạn thấy bản rank khác ở đâu. Ta dựng lại chuỗi `union` của mục 7.1 và để ý bước `union(1, 4)`: cây gốc `2` có **3** phần tử, lớn hơn cây gốc `0` (**2** phần tử), nên bản `DSU` treo `0` dưới `2` — ngược với bản ngây thơ (vốn treo `2` dưới `0` chỉ vì thứ tự tham số). Bản `DSURank` thấy hai cây cùng rank 1 nên treo tùy ý và tăng rank gốc mới lên 2. (Ở bản rank, `parent[3]` in ra là `0` vì lời gọi `union(0, 3)` cuối cùng đã `find(3)` và nén đường.)

In [19]:
class DSU:
    # Union by size + path compression
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            nxt = self.parent[x]
            self.parent[x] = root
            x = nxt
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:   # luôn để ra là cây LỚN hơn
            ra, rb = rb, ra
        self.parent[rb] = ra                # treo cây nhỏ dưới cây lớn
        self.size[ra] += self.size[rb]
        return True


class DSURank(DSU):
    # Chỉ khác cách gộp: so rank (cận trên độ cao) thay vì size
    def __init__(self, n):
        super().__init__(n)
        self.rank = [0] * n

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:  # hai cây cao bằng nhau -> cây mới cao thêm 1
            self.rank[ra] += 1
        return True


for cls in (DSU, DSURank):
    d = cls(6)
    for a, b in [(0, 1), (2, 3), (3, 4), (1, 4), (0, 3)]:
        d.union(a, b)
    extra = f"rank={d.rank}" if cls is DSURank else f"size={d.size}"
    print(f"{cls.__name__:<8} parent={d.parent} {extra} | độ cao={tree_height(d.parent)}")
    print("         4~1:", d.find(4) == d.find(1), "| 4~5:", d.find(4) == d.find(5))

DSU      parent=[2, 0, 2, 2, 2, 5] size=[2, 1, 5, 1, 1, 1] | độ cao=2
         4~1: True | 4~5: False
DSURank  parent=[0, 0, 0, 0, 2, 5] rank=[2, 0, 1, 0, 0, 0] | độ cao=2
         4~1: True | 4~5: False


### 7.5 Thí nghiệm: đo độ cao cây trước và sau tối ưu

Ta so 4 phiên bản DSU trên `n = 2048` phần tử với 3 kiểu chuỗi `union`:

- **dây xích**: `union(i+1, i)` với `i = 0..n-2` — "kẻ thù" của DSU ngây thơ (mục 7.2).
- **gộp đôi**: gộp từng cặp `(0,1), (2,3), ...`, rồi gộp các cặp thành nhóm 4, nhóm 4 thành nhóm 8... — luôn gộp hai cây **bằng nhau**, "kẻ thù" của union by size: đẩy độ cao lên đúng `log₂ 2048 = 11`.
- **ngẫu nhiên**: `2n` cặp ngẫu nhiên (`random.Random(305)` để kết quả lặp lại được).

Với mỗi phiên bản ta đo: **độ cao** ngay sau khi gộp xong, tổng **bước nhảy** (hops — số lần đi `x = parent[x]` trong `find`) trong lúc gộp, rồi gọi `find` cho **mọi** phần tử một lần, đo tổng bước nhảy và độ cao sau đó.

In [20]:
class DSULab:
    # DSU có công tắc bật/tắt 2 tối ưu và bộ đếm bước nhảy
    def __init__(self, n, by_size, compress):
        self.parent = list(range(n))
        self.size = [1] * n
        self.by_size, self.compress = by_size, compress
        self.hops = 0

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
            self.hops += 1
        if self.compress:
            while self.parent[x] != root:
                nxt = self.parent[x]
                self.parent[x] = root
                x = nxt
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.by_size and self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


def seq_chain(n):
    return [(i + 1, i) for i in range(n - 1)]


def seq_pairs(n):
    out, step = [], 1
    while step < n:
        out += [(i, i + step) for i in range(0, n, 2 * step)]
        step *= 2
    return out


def seq_random(n, seed=305):
    rng = random.Random(seed)
    return [(rng.randrange(n), rng.randrange(n)) for _ in range(2 * n)]


N_LAB = 2048
configs = [("ngây thơ", False, False), ("chỉ nén đường", False, True),
           ("chỉ union by size", True, False), ("size + nén", True, True)]
print(f"{'chuỗi':<11}{'phiên bản':<19}{'cao sau union':>14}{'hops union':>12}{'hops find all':>15}{'cao sau':>9}")
for sname, seq in [("dây xích", seq_chain(N_LAB)), ("gộp đôi", seq_pairs(N_LAB)), ("ngẫu nhiên", seq_random(N_LAB))]:
    for cname, by_size, compress in configs:
        d = DSULab(N_LAB, by_size, compress)
        for a, b in seq:
            d.union(a, b)
        h_union, hops_union = tree_height(d.parent), d.hops
        d.hops = 0
        for v in range(N_LAB):
            d.find(v)
        print(f"{sname:<11}{cname:<19}{h_union:>14}{hops_union:>12,}{d.hops:>15,}{tree_height(d.parent):>9}")

chuỗi      phiên bản           cao sau union  hops union  hops find all  cao sau


dây xích   ngây thơ                     2047           0      2,096,128     2047
dây xích   chỉ nén đường                2047           0          4,093        1
dây xích   chỉ union by size               1       2,045          2,047        1
dây xích   size + nén                      1       2,045          2,047        1
gộp đôi    ngây thơ                       11           0         11,264       11
gộp đôi    chỉ nén đường                  11           0          4,083        1
gộp đôi    chỉ union by size              11           0         11,264       11
gộp đôi    size + nén                     11           0          4,083        1


ngẫu nhiên ngây thơ                      357     919,572        484,491      357
ngẫu nhiên chỉ nén đường                   4      16,390          3,842        1
ngẫu nhiên chỉ union by size               5      10,737          3,801        5
ngẫu nhiên size + nén                      2       7,049          2,093        1


### 7.6 Đọc kết quả thí nghiệm

- **Ngây thơ** thảm họa trên dây xích: cây cao `2047`, và `find` cho mọi phần tử tốn khoảng `n²/2 ≈ 2 triệu` bước. Trên dữ liệu ngẫu nhiên cây vẫn cao hàng trăm.
- **Chỉ union by size**: dây xích biến thành cây cao 1 (phần tử mới luôn bị treo dưới cây lớn). Nhưng chuỗi "gộp đôi" đẩy độ cao lên **đúng 11 = log₂ 2048** — khớp với giới hạn `log₂ n` đã chứng minh, và cũng cho thấy giới hạn đó **không thể tốt hơn** nếu chỉ dùng union by size.
- **Chỉ nén đường**: cây vẫn có thể cao lúc mới gộp, nhưng chỉ cần `find` một lượt là gần như mọi thứ dẹt xuống độ cao 1. Cái giá cao được "trả một lần".
- **Cả hai** (size + nén): tổng số bước nhảy (lúc gộp + lúc `find`) nhỏ nhất hoặc ngang bằng nhỏ nhất ở **mọi** chuỗi, và sau một lượt `find` thì cây dẹt hẳn (độ cao 1). Trên dữ liệu ngẫu nhiên, tổng chỉ khoảng `9 000` bước so với gần `1.4 triệu` của bản ngây thơ.

Tóm tắt lý thuyết (khấu hao — amortized — nghĩa là **tính trung bình trên cả dãy thao tác**: một lần lẻ có thể chậm nhưng tổng thì nhỏ):

| Phiên bản | Chi phí mỗi `find` / `union` |
|---|---|
| Ngây thơ | `O(n)` xấu nhất |
| Chỉ union by size (hoặc rank) | `O(log n)` xấu nhất |
| Chỉ path compression | khoảng `O(log n)` khấu hao |
| Union by size/rank **+** path compression | `O(α(n))` khấu hao |

`α(n)` là **hàm Ackermann ngược** (inverse Ackermann) — tăng chậm tới mức `α(n) ≤ 4` với mọi `n` có thể gặp trong thực tế (kể cả `n` bằng số nguyên tử trong vũ trụ). Vì vậy người ta hay nói DSU đầy đủ chạy "**gần như `O(1)`**" mỗi thao tác.

## 8. Cách 3 — Union-Find cho Number of Islands II (lời giải chính)

### 8.1 Ráp DSU vào bài toán

| Trong DSU | Trong bài toán |
|---|---|
| phần tử | một ô, đánh số `id = r * n + c` |
| một tập (một cây) | một **đảo** |
| `parent[id] == -1` | ô `id` còn là **nước** (chưa tham gia DSU) |
| số tập hiện có | `count` = số đảo |
| `union(a, b)` trả `True` | hai đảo **khác nhau** vừa được nối thành một |

Mỗi thao tác thêm ô `(r, c)`:

```text
cur = r * n + c
nếu parent[cur] != -1:           # ô trùng: đã là đất
    ghi count, sang thao tác kế
parent[cur] = cur, size[cur] = 1  # một tập mới chỉ có cur
count += 1                        # thêm một đảo
với mỗi hàng xóm (nr, nc) TRONG lưới và là ĐẤT:
    nếu union(cur, nr * n + nc):  # nối được hai đảo khác nhau
        count -= 1
ghi count
```

**Vì sao "union thành công thì `count -= 1`" khớp đúng công thức `count + 1 − t` ở mục 5?** Giả sử ô mới chạm `t` đảo khác nhau. Hàng xóm đầu tiên thuộc đảo `A` → `union` thành công (ô mới đang là một tập riêng). Từ giờ ô mới và `A` chung gốc, nên mọi hàng xóm **khác** cũng thuộc `A` sẽ cho `union` **thất bại**. Mỗi đảo khác nhau vì thế được gộp **đúng một lần** → đúng `t` lần thành công → `count` cuối = `count cũ + 1 − t`. DSU tự lo chuyện "khác nhau" — ta không cần `set` nhãn như Cách 2.

Quy ước khi hai cây **bằng size**: gốc của **hàng xóm** bị treo dưới gốc của **ô mới** (vì `union(cur, nb)` chỉ đổi chỗ khi `size[ra] < size[rb]`). Tham số `verbose=True` in trace từng bước.

In [21]:
def num_islands2_dsu(m, n, positions, verbose=False):
    parent = [-1] * (m * n)        # -1 = nước
    size = [0] * (m * n)

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:   # path compression
            nxt = parent[x]
            parent[x] = root
            x = nxt
        return root

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:    # union by size
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        return True

    count = 0
    answer = []
    for step, (r, c) in enumerate(positions, 1):
        cur = r * n + c
        if parent[cur] != -1:                       # ô trùng
            answer.append(count)
            if verbose:
                print(f"#{step} ({r},{c}) id={cur}: đã là đất -> bỏ qua, count={count}")
            continue
        parent[cur] = cur
        size[cur] = 1
        count += 1
        if verbose:
            print(f"#{step} ({r},{c}) id={cur}: tập mới, count={count}")
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1:
                nb = nr * n + nc
                if verbose:
                    ra, rb = find(cur), find(nb)
                merged = union(cur, nb)
                if merged:
                    count -= 1
                if verbose:
                    verdict = "gộp" if merged else "CÙNG ĐẢO, bỏ qua"
                    print(f"      hàng xóm id={nb}: gốc {ra} vs gốc {rb} -> {verdict}, count={count}")
        answer.append(count)
        if verbose:
            land = " ".join(f"{i}→{parent[i]}" for i in range(m * n) if parent[i] != -1)
            print(f"      parent: {land}")
    return answer


for name, ex, expected in [("EX1", EX1, [1, 1, 2, 3]), ("EX2", EX2, [1]), ("DUP", DUP, [1, 1, 2, 2]), ("TRACE", TRACE, TRACE_ANSWER)]:
    got = num_islands2_dsu(*ex)
    print(f"{name:<6} {got}")
    assert got == expected

EX1    [1, 1, 2, 3]
EX2    [1]
DUP    [1, 1, 2, 2]
TRACE  [1, 1, 2, 3, 4, 4, 4, 2, 1]


### 8.2 Trace bảng `parent`

**Example 1** (`m = n = 3`) khá hiền: chỉ có **một** lần `union` ở bước 2 (`(0,1)` id 1 chạm `(0,0)` id 0; hai cây cùng size 1 nên gốc của hàng xóm là `0` bị treo dưới `1`). `parent` của các ô đất sau từng bước: `{0→0}` → `{0→1, 1→1}` → thêm `5→5` → thêm `7→7`; đáp án `[1, 1, 2, 3]`.

**Ví dụ `TRACE`** (`m = 3, n = 4`, id của ô `(r, c)` là `4r + c`) mới thật sự thú vị. Hướng xét hàng xóm luôn theo thứ tự `DIRS`: lên, xuống, trái, phải.

| Bước | Ô (id) | Hàng xóm là đất (id) | Kết quả `union` | `count` | `parent` của ô đất sau bước |
|---:|---|---|---|---|---|
| 1 | `(0,0)` id 0 | — | — | 0+1 = 1 | `0→0` |
| 2 | `(0,1)` id 1 | 0 | gộp (size bằng nhau → `0` treo dưới `1`) | 1+1−1 = 1 | `0→1 1→1` |
| 3 | `(2,0)` id 8 | — | — | 2 | `0→1 1→1 8→8` |
| 4 | `(1,3)` id 7 | — | — | 3 | `... 7→7 8→8` |
| 5 | `(2,2)` id 10 | — (`(1,3)` chỉ chạm chéo) | — | 4 | `... 10→10` |
| 6 | `(0,1)` id 1 | — | **trùng**, bỏ qua | 4 | không đổi |
| 7 | `(1,1)` id 5 | 1 | gộp: cây gốc `1` (size 2) lớn hơn → `5→1` | 4+1−1 = 4 | `0→1 1→1 5→1 7→7 8→8 10→10` |
| 8 | `(2,1)` id 9 | 5, 8, 10 | gộp, gộp, gộp (gốc `1` luôn lớn hơn) | 4+1−3 = 2 | `0→1 1→1 5→1 7→7 8→1 9→1 10→1` |
| 9 | `(1,2)` id 6 | 10, 5, 7 | gộp, **cùng đảo**, gộp | 2+1−2 = 1 | `0→1 1→1 5→1 6→1 7→1 8→1 9→1 10→1` |

Ở bước 9, sau khi gộp với id 10, ô mới đã có gốc `1`; hàng xóm id 5 cũng có gốc `1` → `union` trả `False`, `count` **không** bị trừ. Chạy để đối chiếu từng dòng (dòng "gốc A vs gốc B" là gốc của ô mới và gốc của hàng xóm **trước** khi `union`):

In [22]:
print("=== EX1 ===")
num_islands2_dsu(*EX1, verbose=True)
print("=== TRACE ===")
result = num_islands2_dsu(*TRACE, verbose=True)
print("answer =", result)

=== EX1 ===
#1 (0,0) id=0: tập mới, count=1
      parent: 0→0
#2 (0,1) id=1: tập mới, count=2
      hàng xóm id=0: gốc 1 vs gốc 0 -> gộp, count=1
      parent: 0→1 1→1
#3 (1,2) id=5: tập mới, count=2
      parent: 0→1 1→1 5→5
#4 (2,1) id=7: tập mới, count=3
      parent: 0→1 1→1 5→5 7→7
=== TRACE ===
#1 (0,0) id=0: tập mới, count=1
      parent: 0→0
#2 (0,1) id=1: tập mới, count=2
      hàng xóm id=0: gốc 1 vs gốc 0 -> gộp, count=1
      parent: 0→1 1→1
#3 (2,0) id=8: tập mới, count=2
      parent: 0→1 1→1 8→8
#4 (1,3) id=7: tập mới, count=3
      parent: 0→1 1→1 7→7 8→8
#5 (2,2) id=10: tập mới, count=4
      parent: 0→1 1→1 7→7 8→8 10→10
#6 (0,1) id=1: đã là đất -> bỏ qua, count=4
#7 (1,1) id=5: tập mới, count=5
      hàng xóm id=1: gốc 5 vs gốc 1 -> gộp, count=4
      parent: 0→1 1→1 5→1 7→7 8→8 10→10
#8 (2,1) id=9: tập mới, count=5
      hàng xóm id=5: gốc 9 vs gốc 1 -> gộp, count=4
      hàng xóm id=8: gốc 1 vs gốc 8 -> gộp, count=3
      hàng xóm id=10: gốc 1 vs gốc 10 -> gộp, cou

### 8.3 Xử lý ô trùng — chuyện gì xảy ra nếu quên?

Dòng `if parent[cur] != -1: ... continue` trông vô hại nhưng **bắt buộc**. Nếu bỏ nó, một ô đã là đất bị "thêm lại" sẽ:

1. bị gán `parent[cur] = cur`, `size[cur] = 1` — nếu `cur` đang là **gốc** của một đảo lớn thì `size` của cả đảo bị ghi đè thành 1 (hỏng thông tin union by size); nếu `cur` không phải gốc thì nó cùng các phần tử bên dưới bị **tách** khỏi cây;
2. làm `count += 1`, như thể có đảo mới;
3. rồi các `union` với hàng xóm: nếu `cur` vẫn là gốc của chính đảo đó, mọi hàng xóm đều "cùng đảo" → **không** lần nào trừ lại → `count` dư 1 vĩnh viễn.

Ví dụ nhỏ nhất: `m = n = 1`, `positions = [[0,0],[0,0]]` → đúng là `[1, 1]`, bản quên kiểm tra cho `[1, 2]`.

In [23]:
def num_islands2_dsu_no_dup_check(m, n, positions):
    # BẢN SAI: không kiểm tra ô đã là đất
    parent = [-1] * (m * n)
    size = [0] * (m * n)

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        return True

    count, answer = 0, []
    for r, c in positions:
        cur = r * n + c
        parent[cur] = cur                 # <- thiếu "if parent[cur] != -1: ..."
        size[cur] = 1
        count += 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1:
                if union(cur, nr * n + nc):
                    count -= 1
        answer.append(count)
    return answer


for name, ex in [("1x1 trùng", (1, 1, [[0, 0], [0, 0]])), ("DUP", DUP), ("TRACE", TRACE)]:
    print(f"{name:<10} đúng={num_islands2_dsu(*ex)}  quên kiểm tra={num_islands2_dsu_no_dup_check(*ex)}")

1x1 trùng  đúng=[1, 1]  quên kiểm tra=[1, 2]
DUP        đúng=[1, 1, 2, 2]  quên kiểm tra=[1, 1, 2, 3]
TRACE      đúng=[1, 1, 2, 3, 4, 4, 4, 2, 1]  quên kiểm tra=[1, 1, 2, 3, 4, 5, 5, 3, 2]


### 8.4 Độ phức tạp Cách 3

Đặt `N = m·n`, `k = len(positions)`.

- Khởi tạo hai mảng `parent`, `size` dài `N`: `O(N)`.
- Mỗi thao tác: kiểm tra trùng `O(1)`, tối đa 4 lần `union`, mỗi `union` gồm 2 lần `find` → `O(α(N))` khấu hao.
- **Thời gian: `O(N + k · α(N))`** — tốt hơn cả yêu cầu follow-up `O(k log(mn))`. (Nếu chỉ dùng union by size, không nén đường, vẫn đạt đúng `O(N + k log N)`.)
- **Bộ nhớ: `O(N)`** cho `parent` và `size` (không tính mảng kết quả dài `k`).

So với Cách 2: cùng tinh thần "gộp nhỏ vào lớn", nhưng gộp chỉ sửa **một** con trỏ thay vì đổi nhãn cả đảo, và không cần BFS.

## 9. Cách 4 — DSU dùng `dict`, chỉ lưu ô đất

Mảng `parent` dài `m·n` ổn ở bài này vì đề cho `m·n ≤ 10^4`. Nhưng một câu hỏi phỏng vấn tiếp theo rất hay gặp là: *"Nếu lưới khổng lồ, ví dụ `10^9 × 10^9`, mà chỉ có `k = 10^4` thao tác thì sao?"* Mảng `10^18` phần tử là bất khả thi.

**Nhận xét:** chỉ **ô đất** mới cần có mặt trong DSU. Ô nước không bao giờ được `find` hay `union`. Vậy thay mảng bằng `dict`:

- `parent = {}` với key là tuple `(r, c)` (hoặc `id`), value là cha của ô đó.
- **`cell in parent`** đồng thời trả lời "ô này là đất chưa?" — vừa dùng để **kiểm tra trùng**, vừa dùng để **lọc hàng xóm là đất**.
- Bonus: vì key là tuple `(r, c)`, hàng xóm `(0, -1)` hay `(-1, 5)` đơn giản là **không có trong `dict`** (mọi ô trong `positions` đều nằm trong lưới) → không còn bẫy "chỉ số âm quay vòng" hay "tràn sang hàng kế" của mục 3.2. Ta vẫn giữ kiểm tra biên cho rõ nghĩa, nhưng bỏ đi cũng không sai.

Độ phức tạp: **thời gian `O(k · α(k))`** khấu hao (không còn phần khởi tạo `O(N)`), **bộ nhớ `O(k)`** — chỉ phụ thuộc số ô đất, không phụ thuộc kích thước lưới. Đổi lại, mỗi thao tác `dict` (băm tuple) chậm hơn truy cập mảng một hằng số lần.

In [24]:
def num_islands2_dsu_dict(m, n, positions):
    parent = {}                    # chỉ chứa ô ĐẤT: (r, c) -> cha
    size = {}

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            nxt = parent[x]
            parent[x] = root
            x = nxt
        return root

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        return True

    count = 0
    answer = []
    for r, c in positions:
        cell = (r, c)
        if cell in parent:                         # ô trùng
            answer.append(count)
            continue
        parent[cell] = cell
        size[cell] = 1
        count += 1
        for dr, dc in DIRS:
            nb = (r + dr, c + dc)
            if 0 <= nb[0] < m and 0 <= nb[1] < n and nb in parent:
                if union(cell, nb):
                    count -= 1
        answer.append(count)
    return answer


for name, ex, expected in [("EX1", EX1, [1, 1, 2, 3]), ("EX2", EX2, [1]), ("DUP", DUP, [1, 1, 2, 2]), ("TRACE", TRACE, TRACE_ANSWER)]:
    got = num_islands2_dsu_dict(*ex)
    print(f"{name:<6} {got}")
    assert got == expected

BIG = 10**9                                         # lưới 10^9 x 10^9: mảng m*n là bất khả thi
huge = [[0, 0], [0, 1], [BIG - 1, BIG - 1], [1, 1], [1, 0]]
print("Lưới 10^9 x 10^9:", num_islands2_dsu_dict(BIG, BIG, huge))   # [1, 1, 2, 2, 2]
assert num_islands2_dsu_dict(BIG, BIG, huge) == [1, 1, 2, 2, 2]

EX1    [1, 1, 2, 3]
EX2    [1]
DUP    [1, 1, 2, 2]
TRACE  [1, 1, 2, 3, 4, 4, 4, 2, 1]
Lưới 10^9 x 10^9: [1, 1, 2, 2, 2]


## 10. Code nộp LeetCode

### 10.1 Bản chính — DSU mảng (union by size + path compression)

Khác với hàm demo `num_islands2_dsu`: bỏ hết `print`, dùng đúng signature của LeetCode. Trên LeetCode, `List` đã được import sẵn; trong notebook ta đã `from typing import List` ở đầu.

### 10.2 Bản phụ — DSU `dict`

Để ở cell thứ hai dưới tên `SolutionDict` (khi nộp thì đổi tên lại thành `Solution`). Đây là bản nên trình bày nếu người phỏng vấn hỏi tiếp về lưới khổng lồ.

### 10.3 Edge case cần test

| Nhóm | Input | Mong đợi | Kiểm tra điều gì |
|---|---|---|---|
| Ví dụ đề | Example 1, Example 2 | `[1,1,2,3]`, `[1]` | đúng đề |
| Ô trùng | `DUP`; `1x1` thêm `(0,0)` 3 lần | `[1,1,2,2]`; `[1,1,1]` | không đếm đảo "ma" |
| Một hàng / một cột | `1x5` và `5x1`, thêm xen kẽ | `[1,2,3,2,1]` | biên chỉ có 1 chiều |
| Chữ thập | `3x3`, 4 ô quanh tâm rồi tâm | `[1,2,3,4,1]` | một ô gộp 4 đảo (`−3`) |
| Chỉ chạm chéo | `2x2`: `(0,0)`, `(1,1)`, `(0,1)` | `[1,2,1]` | chéo không nối |
| Bàn cờ | `3x3`: 5 ô "đen" trước, 4 ô "trắng" sau | `[1,2,3,4,5,3,2,1,1]` | nhiều hàng xóm cùng đảo |
| Bẫy tràn hàng | `2x3`: `(0,2)`, `(1,0)` | `[1,2]` | ô cuối hàng 0 không kề ô đầu hàng 1 |
| Bẫy chỉ số âm | `3x3`: `(2,0)`, `(0,0)` | `[1,2]` | hàng `-1` không phải hàng cuối |

In [25]:
class Solution:
    def numIslands2(self, m: int, n: int, positions: List[List[int]]) -> List[int]:
        parent = [-1] * (m * n)          # -1 = nước
        size = [0] * (m * n)

        def find(x: int) -> int:
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:     # path compression
                nxt = parent[x]
                parent[x] = root
                x = nxt
            return root

        def union(a: int, b: int) -> bool:
            ra, rb = find(a), find(b)
            if ra == rb:
                return False
            if size[ra] < size[rb]:      # union by size
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]
            return True

        count = 0
        answer = []
        for r, c in positions:
            cur = r * n + c
            if parent[cur] != -1:        # ô trùng: đã là đất
                answer.append(count)
                continue
            parent[cur] = cur
            size[cur] = 1
            count += 1
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1:
                    if union(cur, nr * n + nc):
                        count -= 1
            answer.append(count)
        return answer

In [26]:
EDGE_CASES = [
    ("Example 1",       3, 3, [[0, 0], [0, 1], [1, 2], [2, 1]],                 [1, 1, 2, 3]),
    ("Example 2",       1, 1, [[0, 0]],                                         [1]),
    ("DUP",             3, 3, [[0, 0], [0, 1], [1, 2], [1, 2]],                 [1, 1, 2, 2]),
    ("1x1 trùng 3 lần", 1, 1, [[0, 0], [0, 0], [0, 0]],                         [1, 1, 1]),
    ("một hàng",        1, 5, [[0, 0], [0, 2], [0, 4], [0, 1], [0, 3]],         [1, 2, 3, 2, 1]),
    ("một cột",         5, 1, [[0, 0], [2, 0], [4, 0], [1, 0], [3, 0]],         [1, 2, 3, 2, 1]),
    ("chữ thập",        3, 3, [[0, 1], [1, 0], [1, 2], [2, 1], [1, 1]],         [1, 2, 3, 4, 1]),
    ("chỉ chạm chéo",   2, 2, [[0, 0], [1, 1], [0, 1]],                         [1, 2, 1]),
    ("bàn cờ",          3, 3, [[0, 0], [0, 2], [1, 1], [2, 0], [2, 2],
                               [0, 1], [1, 0], [1, 2], [2, 1]],                 [1, 2, 3, 4, 5, 3, 2, 1, 1]),
    ("bẫy tràn hàng",   2, 3, [[0, 2], [1, 0]],                                 [1, 2]),
    ("bẫy chỉ số âm",   3, 3, [[2, 0], [0, 0]],                                 [1, 2]),
    ("TRACE",           *TRACE,                                                  TRACE_ANSWER),
]

sol = Solution()
for name, m, n, pos, expected in EDGE_CASES:
    got = sol.numIslands2(m, n, pos)
    assert got == expected, (name, got, expected)
    assert num_islands2_recount(m, n, pos) == expected, name     # trọng tài cũng đồng ý
    print(f"{name:<16} {str(got):<28} PASS")

Example 1        [1, 1, 2, 3]                 PASS
Example 2        [1]                          PASS
DUP              [1, 1, 2, 2]                 PASS
1x1 trùng 3 lần  [1, 1, 1]                    PASS
một hàng         [1, 2, 3, 2, 1]              PASS
một cột          [1, 2, 3, 2, 1]              PASS
chữ thập         [1, 2, 3, 4, 1]              PASS
chỉ chạm chéo    [1, 2, 1]                    PASS
bàn cờ           [1, 2, 3, 4, 5, 3, 2, 1, 1]  PASS
bẫy tràn hàng    [1, 2]                       PASS
bẫy chỉ số âm    [1, 2]                       PASS
TRACE            [1, 1, 2, 3, 4, 4, 4, 2, 1]  PASS


In [27]:
class SolutionDict:
    # Bản phụ: DSU dùng dict, chỉ lưu ô đất (khi nộp đổi tên thành Solution)
    def numIslands2(self, m: int, n: int, positions: List[List[int]]) -> List[int]:
        parent = {}
        size = {}

        def find(x):
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                nxt = parent[x]
                parent[x] = root
                x = nxt
            return root

        def union(a, b) -> bool:
            ra, rb = find(a), find(b)
            if ra == rb:
                return False
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            parent[rb] = ra
            size[ra] += size[rb]
            return True

        count = 0
        answer = []
        for r, c in positions:
            cell = (r, c)
            if cell in parent:
                answer.append(count)
                continue
            parent[cell] = cell
            size[cell] = 1
            count += 1
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                nb = (r + dr, c + dc)
                if nb in parent and union(cell, nb):   # ngoài lưới thì chắc chắn không có trong dict
                    count -= 1
            answer.append(count)
        return answer


sol_dict = SolutionDict()
for name, m, n, pos, expected in EDGE_CASES:
    assert sol_dict.numIslands2(m, n, pos) == expected, name
print("SolutionDict: tất cả", len(EDGE_CASES), "edge case PASS")

SolutionDict: tất cả 12 edge case PASS


### 10.4 Test đối chiếu chéo trên input ngẫu nhiên, và thử ở giới hạn đề

**Đối chiếu chéo:** sinh 400 bộ input nhỏ (`1 ≤ m, n ≤ 6`, `k` tới `2·m·n` nên gần như chắc chắn có ô trùng) bằng `random.Random(305)` — seed cố định để lần chạy nào cũng ra cùng dữ liệu. Mọi cách (2, 3, 4, `Solution`, `SolutionDict`) phải cho **đúng y** kết quả của trọng tài Cách 1. Ta cũng đếm xem dữ liệu có thật sự chứa các tình huống khó (ô trùng, một ô gộp ≥ 2 đảo) không — test ngẫu nhiên chỉ có giá trị nếu nó chạm tới các nhánh đặc biệt.

**Giới hạn đề:** `m·n = 10^4`, `k = 10^4` với vài hình dạng: lưới vuông `100x100` ngẫu nhiên, một hàng `1x10000`, "cái lược" `3x3333`, và `1x1` với `10^4` ô trùng. Cách 1 không chạy ở đây (đã ước lượng hàng chục giây trở lên ở mục 4).

In [28]:
rng = random.Random(305)
methods = {
    "relabel": num_islands2_relabel,
    "dsu": num_islands2_dsu,
    "dsu_dict": num_islands2_dsu_dict,
    "Solution": Solution().numIslands2,
    "SolutionDict": SolutionDict().numIslands2,
}
with_dup = multi_merge = 0
for trial in range(400):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    k = rng.randint(1, 2 * m * n)
    pos = [[rng.randrange(m), rng.randrange(n)] for _ in range(k)]
    ref = num_islands2_recount(m, n, pos)
    for name, f in methods.items():
        assert f(m, n, pos) == ref, (name, m, n, pos)
    with_dup += len({tuple(p) for p in pos}) < len(pos)
    multi_merge += any(b < a for a, b in zip(ref, ref[1:]))   # số đảo giảm = có bước gộp >= 2 đảo
print("400 test ngẫu nhiên: mọi cách khớp trọng tài Cách 1")
print(f"  - số test có ô trùng          : {with_dup}")
print(f"  - số test có bước gộp >= 2 đảo: {multi_merge}")

400 test ngẫu nhiên: mọi cách khớp trọng tài Cách 1
  - số test có ô trùng          : 323
  - số test có bước gộp >= 2 đảo: 214


In [29]:
def timeit(f, *args):
    t0 = time.perf_counter()
    out = f(*args)
    return time.perf_counter() - t0, out


rng = random.Random(305)
big_cases = [
    ("100x100 ngẫu nhiên", 100, 100, [[rng.randrange(100), rng.randrange(100)] for _ in range(10**4)]),
    ("1x10000 hoán vị",    1, 10**4, [[0, c] for c in rng.sample(range(10**4), 10**4)]),
    ("3x3333 cái lược",    3, 3333, comb_positions(3333)),
    ("1x1 trùng 10^4 lần", 1, 1, [[0, 0]] * 10**4),
]
print(f"{'input':<20}{'k':>7}{'relabel':>10}{'dsu':>9}{'dsu_dict':>10}{'Solution':>10}")
for name, m, n, pos in big_cases:
    t_rel, a_rel = timeit(num_islands2_relabel, m, n, pos)
    t_dsu, a_dsu = timeit(num_islands2_dsu, m, n, pos)
    t_dic, a_dic = timeit(num_islands2_dsu_dict, m, n, pos)
    t_sol, a_sol = timeit(Solution().numIslands2, m, n, pos)
    assert a_rel == a_dsu == a_dic == a_sol
    print(f"{name:<20}{len(pos):>7}{t_rel:>9.3f}s{t_dsu:>8.3f}s{t_dic:>9.3f}s{t_sol:>9.3f}s")

input                     k   relabel      dsu  dsu_dict  Solution
100x100 ngẫu nhiên    10000    0.017s   0.008s    0.011s    0.008s


1x10000 hoán vị       10000    0.033s   0.009s    0.016s    0.009s


3x3333 cái lược        6667    0.010s   0.007s    0.009s    0.006s
1x1 trùng 10^4 lần    10000    0.000s   0.001s    0.001s    0.000s


## 11. Vì sao lời giải đúng?

### 11.1 Bất biến (invariant) của Cách 3

Sau **mỗi** thao tác, ba điều sau luôn đúng:

1. Ô `id` là đất **⇔** `parent[id] != -1`.
2. Hai ô đất có **cùng gốc** trong DSU **⇔** chúng thuộc **cùng một đảo** (nối được với nhau qua các ô đất kề 4 hướng).
3. `count` = số cây trong DSU = **số đảo**.

**Chứng minh bằng quy nạp** (đúng ở bước trước ⇒ đúng ở bước sau):

- *Ban đầu*: lưới toàn nước, `parent` toàn `-1`, `count = 0` — đúng cả ba.
- *Thao tác thêm ô trùng*: lưới không đổi, ta không đổi gì → vẫn đúng.
- *Thao tác thêm ô mới `x`*: theo mục 5, các đảo **không chạm** `x` giữ nguyên; các đảo chạm `x` cùng `x` thành **một** đảo. Trong DSU: ta tạo tập `{x}` rồi `union` `x` với **mọi** hàng xóm là đất → tập chứa `x` trở thành `{x}` hợp với **đúng** các tập chạm `x`; các tập khác không bị đụng tới. Vậy (2) đúng. `count` tăng 1 cho tập `{x}` và giảm 1 cho mỗi `union` thành công — mỗi lần như vậy hai tập khác nhau nhập làm một, số cây giảm đúng 1 → (3) đúng.

**Không bỏ sót:** một kết nối mới giữa hai đảo cũ **bắt buộc** đi qua `x`, và bước vào `x` thì phải từ 4 ô kề → chỉ cần xét 4 hàng xóm.
**Không gộp nhầm:** ta chỉ `union` với hàng xóm **trong lưới** và **là đất** → mỗi lần gộp ứng với một cặp ô thật sự kề nhau.

Cách 2 đúng vì cùng lý do, với bất biến "hai ô cùng đảo ⇔ cùng nhãn". Cách 4 giống hệt Cách 3, chỉ đổi mảng thành `dict` (bất biến 1 thành "`cell in parent` ⇔ đất").

## 12. Bảng so sánh các cách

Đặt `N = m·n`, `k = len(positions)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Nhận xét |
|---|---|---|---|---|
| 1. Đếm lại | BFS cả lưới sau mỗi thao tác | `O(k · N)` | `O(N)` | Dễ nhất, đúng hiển nhiên → làm **trọng tài**; TLE ở giới hạn đề |
| 2. Nhãn + đổi nhãn phần nhỏ | Lưới nhãn "sống"; gộp thì BFS đổi nhãn các đảo nhỏ | `O(k + N log N)` | `O(N)` | Đạt; phải chọn đúng đảo lớn (nếu không thành `O(N²)`); code dài |
| 3. DSU mảng (**chính**) | `parent`/`size` dài `N`; `count += 1`, `union` thành công thì `count -= 1` | `O(N + k · α(N))` | `O(N)` | Ngắn, nhanh, vượt follow-up `O(k log N)` |
| 4. DSU `dict` | Chỉ lưu ô đất, key là `(r, c)` | `O(k · α(k))` | `O(k)` | Dùng được cho lưới khổng lồ; hằng số chậm hơn mảng |

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra ô trùng
Đã minh họa ở mục 8.3: `positions = [[0,0],[0,0]]` cho `[1, 2]` thay vì `[1, 1]`. **Sửa:** `if parent[cur] != -1: answer.append(count); continue`.

### Lỗi 2: Trừ `count` theo **số hàng xóm là đất** thay vì theo **số `union` thành công**
```python
if parent[nb] != -1:
    union(cur, nb)
    count -= 1          # SAI: hàng xóm cùng đảo cũng bị trừ
```
Bước 9 của `TRACE` có 3 hàng xóm là đất nhưng chỉ 2 đảo → ra `0` thay vì `1`. **Sửa:** `if union(cur, nb): count -= 1`.

### Lỗi 3: Đánh số ô bằng `r * m + c`
`id` phải nhân với **số cột** `n`. Với `m = 2, n = 3`: `(0,2)` và `(1,0)` cùng ra `id = 2` → ô thứ hai bị coi là "trùng". Lỗi này **không lộ** khi test với lưới vuông `m == n` — luôn test lưới chữ nhật!

### Lỗi 4: Kiểm tra biên thiếu
- Chỉ kiểm tra `0 <= nb_id < m*n`: ô cuối hàng bị coi là kề ô đầu hàng kế (tràn hàng).
- Quên vế `>= 0` (chỉ viết `nr < m and nc < n`): `parent[-3]` lặng lẽ đọc ô ở **hàng cuối** — Python không báo lỗi với chỉ số âm.

**Sửa:** luôn `0 <= nr < m and 0 <= nc < n`, tính `id` **sau** khi kiểm tra.

### Lỗi 5: Gắn **nút** thay vì gắn **gốc**
```python
parent[b] = a           # SAI: chỉ chuyển b, phần còn lại của cây b bị bỏ lại
parent[rb] = ra         # ĐÚNG: gắn gốc vào gốc
```

### Lỗi 6: `continue` khi gặp ô trùng mà quên `answer.append(count)`
Mảng kết quả **ngắn hơn** `positions` → sai format. Mỗi thao tác phải có đúng một phần tử trong `answer`.

### Lỗi 7: `find` đệ quy trên cây cao → `RecursionError`
`find` đệ quy chỉ an toàn khi cây thấp (có union by size/rank). Với DSU ngây thơ, dây xích 5000 phần tử vượt giới hạn ~1000 tầng của Python. **Sửa:** viết `find` bằng vòng lặp hai lượt như notebook này, và luôn dùng union by size/rank.

Cell dưới cài các lỗi vào một hàm `buggy_islands2` (chọn lỗi bằng tham số `bug`) và so với đáp án đúng; sau đó tái hiện Lỗi 7.

In [30]:
def buggy_islands2(m, n, positions, bug):
    parent = [-1] * (m * n)
    size = [0] * (m * n)

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        if bug == "attach_node":
            parent[b] = a                       # Lỗi 5
        else:
            parent[rb] = ra
        size[ra] += size[rb]
        return True

    def to_id(r, c):
        return r * m + c if bug == "id_uses_m" else r * n + c    # Lỗi 3

    def inside(nr, nc):
        if bug == "id_range_only":
            return 0 <= nr * n + nc < m * n                          # Lỗi 4a
        if bug == "forget_lower_bound":
            return nr < m and nc < n                                 # Lỗi 4b
        return 0 <= nr < m and 0 <= nc < n

    count, answer = 0, []
    for r, c in positions:
        cur = to_id(r, c)
        if parent[cur] != -1:
            if bug != "no_append_on_dup":                            # Lỗi 6
                answer.append(count)
            continue
        parent[cur] = cur
        size[cur] = 1
        count += 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if inside(nr, nc) and parent[to_id(nr, nc)] != -1:
                merged = union(cur, to_id(nr, nc))
                if merged or bug == "count_per_neighbor":            # Lỗi 2
                    count -= 1
        answer.append(count)
    return answer


demo = [
    ("Lỗi 2 count_per_neighbor", "count_per_neighbor", TRACE),
    ("Lỗi 3 id_uses_m",          "id_uses_m",          (2, 3, [[0, 2], [1, 0]])),
    ("Lỗi 4a id_range_only",     "id_range_only",      (2, 3, [[0, 2], [1, 0]])),
    ("Lỗi 4b forget_lower_bound", "forget_lower_bound", (3, 3, [[2, 0], [0, 0]])),
    ("Lỗi 5 attach_node",        "attach_node",        TRACE),
    ("Lỗi 6 no_append_on_dup",   "no_append_on_dup",   DUP),
]
for title, bug, ex in demo:
    right, wrong = num_islands2_dsu(*ex), buggy_islands2(*ex, bug)
    print(f"{title:<27} đúng={str(right):<29} sai={wrong}")
    assert right != wrong

# Lỗi 7: find đệ quy trên dây xích dài
def find_recursive(parent, x):
    if parent[x] != x:
        parent[x] = find_recursive(parent, parent[x])
    return parent[x]

chain = list(range(1, 5000)) + [4999]          # 0 -> 1 -> ... -> 4999
try:
    find_recursive(chain, 0)
except RecursionError as e:
    print("Lỗi 7: RecursionError:", str(e)[:45], "...")

Lỗi 2 count_per_neighbor    đúng=[1, 1, 2, 3, 4, 4, 4, 2, 1]   sai=[1, 1, 2, 3, 4, 4, 4, 2, 0]
Lỗi 3 id_uses_m             đúng=[1, 2]                        sai=[1, 1]
Lỗi 4a id_range_only        đúng=[1, 2]                        sai=[1, 1]
Lỗi 4b forget_lower_bound   đúng=[1, 2]                        sai=[1, 1]
Lỗi 5 attach_node           đúng=[1, 1, 2, 3, 4, 4, 4, 2, 1]   sai=[1, 1, 2, 3, 4, 4, 4, 2, 0]
Lỗi 6 no_append_on_dup      đúng=[1, 1, 2, 2]                  sai=[1, 1, 2]
Lỗi 7: RecursionError: maximum recursion depth exceeded ...


## 14. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu "đây là bài Union-Find"

- Có các **phần tử** và các **kết nối được thêm dần theo thời gian** (online — câu hỏi xen kẽ với cập nhật).
- Câu hỏi là **"có bao nhiêu nhóm?"**, **"hai phần tử có cùng nhóm không?"**, hoặc "nhóm lớn nhất bao nhiêu?".
- **Chỉ thêm, không xóa** kết nối. DSU không hỗ trợ "tách" một tập.
- Nếu đề chỉ **xóa** (ví dụ đất bị chìm dần, gạch bị bắn rơi) → mẹo **đảo ngược thời gian**: xử lý từ trạng thái cuối, "xóa" ngược lại thành "thêm" (xem Bài 6 bên dưới).
- Nếu lưới cố định và chỉ hỏi **một lần** → BFS/DFS (bài #200) là đủ, DSU không cần thiết.

### Câu thần chú

> *"Thêm một ô: `count += 1`. Mỗi lần `union` thành công: `count -= 1`. Trùng thì bỏ qua — nhưng vẫn ghi kết quả."*

### Bài liên quan

| Bài | Độ khó | Liên hệ |
|---|---|---|
| #200 Number of Islands | Medium | Phiên bản tĩnh — BFS/DFS loang (đã ôn ở mục 3) |
| #323 Number of Connected Components in an Undirected Graph | Medium (Premium) | DSU trên đồ thị cạnh: `count = n`, mỗi `union` thành công trừ 1 (Bài 5) |
| #547 Number of Provinces | Medium | Như #323 nhưng đồ thị cho bằng ma trận kề |
| #684 Redundant Connection | Medium | Cạnh làm `union` **thất bại** chính là cạnh tạo chu trình |
| #721 Accounts Merge | Medium | DSU trên email (dùng `dict` như Cách 4) |
| #827 Making A Large Island | Hard | Gán nhãn/DSU + `size`, thử lật một ô nước |
| #803 Bricks Falling When Hit | Hard | Chỉ xóa → đảo ngược thời gian + DSU |
| #1970 Last Day Where You Can Still Cross | Hard | Thêm dần ô (hoặc đảo ngược) + DSU, hỏi khi nào hai biên nối nhau |
| #778 Swim in Rising Water | Hard | Thêm ô theo thứ tự độ cao + DSU |

## 15. Bài tập vận dụng

Làm theo thứ tự, tự làm trước rồi mới xem gợi ý ở mục 16 và đáp án ở mục 17. Các cell TODO chỉ **định nghĩa** hàm (chưa gọi); làm xong thì bỏ comment các dòng `assert` để tự kiểm tra.

### Bài 1 — Tính tay (không chạy code)

`m = 2, n = 3, positions = [[0,0],[1,1],[0,2],[0,1],[1,1],[1,0]]`.

1. Vẽ lưới sau mỗi thao tác và ghi số đảo → viết ra `answer` (6 số).
2. Với DSU của Cách 3 (xét hàng xóm theo thứ tự lên, xuống, trái, phải), ở **bước 4** và **bước 6** có bao nhiêu lần `union` thành công, bao nhiêu lần thất bại?
3. Bước nào là ô trùng?

Đáp án được kiểm chứng bằng code ở mục 17.

### Bài 2 — Điền TODO: `find` có nén đường và `union` theo size

Hoàn thiện class dưới. `union` phải trả về `True` nếu thật sự gộp hai tập khác nhau, `False` nếu đã cùng tập.

In [31]:
class ExerciseDSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        # TODO 1: lượt 1 - đi lên tới gốc
        # TODO 2: lượt 2 - cho mọi nút trên đường trỏ thẳng vào gốc
        pass

    def union(self, a, b):
        # TODO 3: tìm gốc ra, rb; nếu trùng thì trả False
        # TODO 4: để ra là gốc của cây LỚN hơn, treo rb dưới ra, cập nhật size
        pass

# Gợi ý test sau khi làm xong:
# d = ExerciseDSU(6)
# assert d.union(0, 1) is True and d.union(2, 3) is True
# assert d.union(1, 0) is False
# assert d.union(1, 3) is True and d.find(0) == d.find(2)
# assert d.size[d.find(0)] == 4

### Bài 3 — Tự viết lại lời giải chính

Viết `exercise_num_islands2(m, n, positions)` bằng DSU mảng, **không nhìn** code ở mục 10. Nhớ đủ 4 chi tiết: `id = r * n + c`, kiểm tra trùng (và vẫn ghi kết quả), kiểm tra biên **hai chiều**, chỉ trừ `count` khi `union` thành công.

In [32]:
def exercise_num_islands2(m: int, n: int, positions: List[List[int]]) -> List[int]:
    # TODO: parent = [-1] * (m * n), size, count, answer
    # TODO: với mỗi (r, c): xử lý ô trùng, tạo tập mới, union với hàng xóm là đất
    raise NotImplementedError

# Gợi ý test:
# for name, m, n, pos, expected in EDGE_CASES:
#     assert exercise_num_islands2(m, n, pos) == expected, name

### Bài 4 — Biến thể: diện tích đảo lớn nhất sau mỗi thao tác

Thay vì số đảo, trả về `best[i]` = **số ô của đảo lớn nhất** sau thao tác `i`. Ví dụ `EX1` → `[1, 2, 2, 2]`; `TRACE` → `[1, 2, 2, 2, 2, 2, 3, 6, 8]`.

Gợi ý nhỏ: `size[find(x)]` cho biết kích thước đảo chứa `x`. Đảo chỉ có thể **lớn lên** (không bao giờ bị xóa), nên giá trị lớn nhất không bao giờ giảm.

In [33]:
def max_island_after_each(m: int, n: int, positions: List[List[int]]) -> List[int]:
    # TODO: DSU như Bài 3, thêm biến best; sau mỗi thao tác best = max(best, size[find(cur)])
    raise NotImplementedError

# Gợi ý test:
# assert max_island_after_each(*EX1) == [1, 2, 2, 2]
# assert max_island_after_each(*TRACE) == [1, 2, 2, 2, 2, 2, 3, 6, 8]
# assert max_island_after_each(*DUP) == [1, 2, 2, 2]

### Bài 5 — Chuyển giao: #323 Number of Connected Components in an Undirected Graph

Cho `n` đỉnh đánh số `0..n-1` và danh sách cạnh vô hướng `edges` (`edges[i] = [a, b]`). Trả về số **thành phần liên thông**. Signature LeetCode: `def countComponents(self, n: int, edges: List[List[int]]) -> int`.

- `n = 5, edges = [[0,1],[1,2],[3,4]]` → `2`
- `n = 5, edges = [[0,1],[1,2],[2,3],[3,4]]` → `1`

Đây chính là "Number of Islands II" nhưng phần tử là đỉnh, kết nối là cạnh: bắt đầu với `n` nhóm, mỗi `union` thành công trừ 1.

In [34]:
def count_components(n: int, edges: List[List[int]]) -> int:
    # TODO: count = n; với mỗi cạnh, nếu union thành công thì count -= 1
    raise NotImplementedError

# Gợi ý test:
# assert count_components(5, [[0, 1], [1, 2], [3, 4]]) == 2
# assert count_components(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == 1
# assert count_components(3, []) == 3

### Bài 6 — Mini challenge: đất chìm dần (đảo ngược thời gian)

Cho lưới `grid` (list lồng list gồm `0/1`) và danh sách `removals` — các ô **đất** (khác nhau) lần lượt bị **chìm** thành nước. Trả về `answer[i]` = số đảo **sau** khi ô `removals[i]` chìm.

Ví dụ:

```text
grid = [[1,1,1],
        [0,1,0],
        [1,1,1]]
removals = [[1,1],[0,1],[2,1]]   ->   [2, 3, 4]
```

(Chìm `(1,1)`: hàng trên và hàng dưới tách nhau → 2. Chìm `(0,1)`: hàng trên tách đôi → 3. Chìm `(2,1)`: hàng dưới tách đôi → 4.)

DSU không biết "tách" tập. Mẹo: đi **ngược** thời gian — bắt đầu từ lưới **sau khi mọi ô đã chìm**, rồi "thêm lại" từng ô theo thứ tự ngược. Yêu cầu: `O((m·n + r) · α(m·n))` với `r = len(removals)`.

In [35]:
def islands_after_removals(grid: List[List[int]], removals: List[List[int]]) -> List[int]:
    # TODO 1: tạo lưới cuối cùng (đã chìm hết các ô trong removals)
    # TODO 2: dựng DSU cho lưới cuối cùng, tính count
    # TODO 3: duyệt i từ cuối về đầu: answer[i] = count, rồi "thêm lại" ô removals[i]
    raise NotImplementedError

# Gợi ý test:
# g = [[1, 1, 1], [0, 1, 0], [1, 1, 1]]
# assert islands_after_removals(g, [[1, 1], [0, 1], [2, 1]]) == [2, 3, 4]

## 16. Gợi ý theo mức

**Bài 1**
- Mức 1: vẽ lưới `2x3` sáu lần; nhớ chỉ nối ngang/dọc.
- Mức 2: bước 2 `(1,1)` chỉ chạm **chéo** `(0,0)`. Bước 4 `(0,1)` có 3 hàng xóm là đất thuộc 3 đảo khác nhau.
- Mức 3: ở bước 6, ô `(1,0)` chạm `(0,0)` (lên) và `(1,1)` (phải) — sau bước 4 hai ô này đã cùng một đảo.

**Bài 2**
- Mức 1: chép đúng cấu trúc "hai lượt" của mục 7.3; `union` giống mục 7.4.
- Mức 2: lượt 2 cần biến tạm `nxt = self.parent[x]` trước khi gán `self.parent[x] = root`.
- Mức 3: `if self.size[ra] < self.size[rb]: ra, rb = rb, ra` rồi `self.parent[rb] = ra; self.size[ra] += self.size[rb]; return True`.

**Bài 3**
- Mức 1: viết khung vòng lặp `for r, c in positions` trước, các hàm `find`/`union` lồng bên trong.
- Mức 2: thứ tự trong vòng lặp: tính `cur` → kiểm tra trùng → `parent[cur] = cur; size[cur] = 1; count += 1` → 4 hàng xóm → `answer.append(count)`.
- Mức 3: điều kiện hàng xóm: `0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1`, rồi `if union(cur, nr * n + nc): count -= 1`.

**Bài 4**
- Mức 1: bắt đầu từ lời giải Bài 3, chỉ thêm một biến `best = 0`.
- Mức 2: với ô trùng, `best` không đổi — vẫn phải `append`.
- Mức 3: sau vòng 4 hàng xóm: `best = max(best, size[find(cur)])`.

**Bài 5**
- Mức 1: không cần lưới, không cần hàng xóm: các phần tử là `0..n-1`, kết nối là các cạnh.
- Mức 2: dùng lại class `DSU` ở mục 7.4.
- Mức 3: `count = n; for a, b in edges: if d.union(a, b): count -= 1`.

**Bài 6**
- Mức 1: nếu xem thời gian chạy ngược, "chìm" trở thành "nổi lên" — đúng bài Number of Islands II.
- Mức 2: `answer[i]` là số đảo ở trạng thái "`removals[0..i]` đã chìm, `removals[i+1..]` còn nguyên". Khi đi ngược (vòng `i` chạy từ `r-1` về `0`), ở **đầu** vòng `i` lưới đang đúng trạng thái đó — vì các ô `removals[i+1..]` vừa được thêm lại ở những vòng trước. Vậy: ghi `answer[i] = count` **trước**, rồi mới thêm lại `removals[i]`.
- Mức 3: lưới cuối: sao chép `grid`, đặt các ô trong `removals` về `0`. Dựng DSU: mỗi ô đất `count += 1` và `union` với hàng xóm trên/trái đã là đất. Phần "thêm lại" dùng y nguyên vòng lặp của Cách 3.

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này. Mỗi đáp án đều có `assert` chạy được.

**Bài 1 (tính tay):** `answer = [1, 2, 3, 1, 1, 1]`.

- Bước 4 `(0,1)`: hàng xóm lên — ngoài lưới; xuống `(1,1)` → `union` thành công; trái `(0,0)` → thành công; phải `(0,2)` → thành công. **3 thành công, 0 thất bại**: `3 + 1 − 3 = 1`.
- Bước 5 `(1,1)`: **trùng**, giữ `1`.
- Bước 6 `(1,0)`: lên `(0,0)` → thành công; xuống, trái — ngoài lưới; phải `(1,1)` → **thất bại** (đã cùng đảo). **1 thành công, 1 thất bại**: `1 + 1 − 1 = 1`.

In [36]:
# ===== Bài 1: kiểm chứng bằng trace =====
BAI1 = (2, 3, [[0, 0], [1, 1], [0, 2], [0, 1], [1, 1], [1, 0]])
assert num_islands2_dsu(*BAI1) == num_islands2_recount(*BAI1) == [1, 2, 3, 1, 1, 1]
print("Bài 1:", num_islands2_dsu(*BAI1, verbose=True))     # xem số union thành công / thất bại ở bước 4, 6


# ===== Bài 2 =====
class AnswerDSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            nxt = self.parent[x]
            self.parent[x] = root
            x = nxt
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


d = AnswerDSU(6)
assert d.union(0, 1) is True and d.union(2, 3) is True
assert d.union(1, 0) is False
assert d.union(1, 3) is True and d.find(0) == d.find(2)
assert d.size[d.find(0)] == 4
print("Bài 2: PASS")


# ===== Bài 3 =====
def answer_num_islands2(m: int, n: int, positions: List[List[int]]) -> List[int]:
    parent = [-1] * (m * n)
    size = [0] * (m * n)

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            nxt = parent[x]
            parent[x] = root
            x = nxt
        return root

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        return True

    count, answer = 0, []
    for r, c in positions:
        cur = r * n + c
        if parent[cur] != -1:
            answer.append(count)
            continue
        parent[cur], size[cur] = cur, 1
        count += 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1:
                if union(cur, nr * n + nc):
                    count -= 1
        answer.append(count)
    return answer


for name, m, n, pos, expected in EDGE_CASES:
    assert answer_num_islands2(m, n, pos) == expected, name
print("Bài 3: PASS trên", len(EDGE_CASES), "edge case")

#1 (0,0) id=0: tập mới, count=1
      parent: 0→0
#2 (1,1) id=4: tập mới, count=2
      parent: 0→0 4→4
#3 (0,2) id=2: tập mới, count=3
      parent: 0→0 2→2 4→4
#4 (0,1) id=1: tập mới, count=4
      hàng xóm id=4: gốc 1 vs gốc 4 -> gộp, count=3
      hàng xóm id=0: gốc 1 vs gốc 0 -> gộp, count=2
      hàng xóm id=2: gốc 1 vs gốc 2 -> gộp, count=1
      parent: 0→1 1→1 2→1 4→1
#5 (1,1) id=4: đã là đất -> bỏ qua, count=1
#6 (1,0) id=3: tập mới, count=2
      hàng xóm id=0: gốc 3 vs gốc 1 -> gộp, count=1
      hàng xóm id=4: gốc 1 vs gốc 1 -> CÙNG ĐẢO, bỏ qua, count=1
      parent: 0→1 1→1 2→1 3→1 4→1
Bài 1: [1, 2, 3, 1, 1, 1]
Bài 2: PASS
Bài 3: PASS trên 12 edge case


In [37]:
# ===== Bài 4 =====
def answer_max_island_after_each(m: int, n: int, positions: List[List[int]]) -> List[int]:
    parent = [-1] * (m * n)
    size = [0] * (m * n)

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            nxt = parent[x]
            parent[x] = root
            x = nxt
        return root

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]
        return True

    best, out = 0, []
    for r, c in positions:
        cur = r * n + c
        if parent[cur] == -1:
            parent[cur], size[cur] = cur, 1
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1:
                    union(cur, nr * n + nc)
            best = max(best, size[find(cur)])
        out.append(best)
    return out


assert answer_max_island_after_each(*EX1) == [1, 2, 2, 2]
assert answer_max_island_after_each(*TRACE) == [1, 2, 2, 2, 2, 2, 3, 6, 8]
assert answer_max_island_after_each(*DUP) == [1, 2, 2, 2]
print("Bài 4: PASS")


# ===== Bài 5 =====
def answer_count_components(n: int, edges: List[List[int]]) -> int:
    d = DSU(n)
    count = n
    for a, b in edges:
        if d.union(a, b):
            count -= 1
    return count


assert answer_count_components(5, [[0, 1], [1, 2], [3, 4]]) == 2
assert answer_count_components(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == 1
assert answer_count_components(3, []) == 3
print("Bài 5: PASS")


# ===== Bài 6 =====
def answer_islands_after_removals(grid: List[List[int]], removals: List[List[int]]) -> List[int]:
    m, n = len(grid), len(grid[0])
    final = [row[:] for row in grid]
    for r, c in removals:
        final[r][c] = 0
    d = DSU(m * n)
    is_land = [[final[r][c] == 1 for c in range(n)] for r in range(m)]
    count = 0
    for r in range(m):                       # dựng DSU cho lưới cuối cùng
        for c in range(n):
            if is_land[r][c]:
                count += 1
                if r > 0 and is_land[r - 1][c] and d.union(r * n + c, (r - 1) * n + c):
                    count -= 1
                if c > 0 and is_land[r][c - 1] and d.union(r * n + c, r * n + c - 1):
                    count -= 1
    answer = [0] * len(removals)
    for i in range(len(removals) - 1, -1, -1):   # đi NGƯỢC thời gian
        answer[i] = count                         # số đảo khi removals[0..i] đã chìm
        r, c = removals[i]
        is_land[r][c] = True                      # "thêm lại" ô removals[i]
        count += 1
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and is_land[nr][nc]:
                if d.union(r * n + c, nr * n + nc):
                    count -= 1
    return answer


g = [[1, 1, 1], [0, 1, 0], [1, 1, 1]]
assert answer_islands_after_removals(g, [[1, 1], [0, 1], [2, 1]]) == [2, 3, 4]

rng = random.Random(305)                          # đối chiếu với cách "chìm rồi đếm lại"
for _ in range(200):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    grid = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    lands = [[r, c] for r in range(m) for c in range(n) if grid[r][c] == 1]
    removals = rng.sample(lands, rng.randint(0, len(lands)))
    cur, expected = [row[:] for row in grid], []
    for r, c in removals:
        cur[r][c] = 0
        expected.append(num_islands_bfs(cur))
    assert answer_islands_after_removals(grid, removals) == expected
print("Bài 6: PASS (ví dụ + 200 test ngẫu nhiên)")

Bài 4: PASS
Bài 5: PASS
Bài 6: PASS (ví dụ + 200 test ngẫu nhiên)


## 18. Template cuối cùng nên nhớ

**DSU chuẩn** (union by size + path compression, `find` không đệ quy):

```python
def find(x):
    root = x
    while parent[root] != root:
        root = parent[root]
    while parent[x] != root:            # nén đường
        nxt = parent[x]; parent[x] = root; x = nxt
    return root

def union(a, b):                        # True nếu thật sự gộp 2 tập khác nhau
    ra, rb = find(a), find(b)
    if ra == rb:
        return False
    if size[ra] < size[rb]:
        ra, rb = rb, ra
    parent[rb] = ra; size[ra] += size[rb]
    return True
```

**Number of Islands II**:

```python
parent = [-1] * (m * n); size = [0] * (m * n); count = 0; answer = []
for r, c in positions:
    cur = r * n + c                     # nhân với SỐ CỘT n
    if parent[cur] != -1:               # ô trùng
        answer.append(count); continue
    parent[cur] = cur; size[cur] = 1; count += 1
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and parent[nr * n + nc] != -1:
            if union(cur, nr * n + nc): # chỉ trừ khi gộp THẬT
                count -= 1
    answer.append(count)
return answer
```

**Câu thần chú**: *"Thêm một ô là thêm một đảo; gộp được bao nhiêu thì trừ bấy nhiêu; ô trùng thì giữ nguyên."*

## 19. Checklist tự kiểm tra trước khi submit

- `id = r * n + c` — nhân với **số cột** `n`, không phải `m`? Đã test với lưới **chữ nhật** (`m != n`) chưa?
- Có kiểm tra **ô trùng** (`parent[cur] != -1`) và **vẫn** `answer.append(count)` trước khi `continue` chưa?
- Kiểm tra biên đủ **cả hai vế, cả hàng lẫn cột**: `0 <= nr < m and 0 <= nc < n` — tính `id` hàng xóm **sau** khi đã kiểm tra?
- Chỉ `count -= 1` khi `union(...)` trả về `True` (hai đảo **khác nhau**), không phải theo số hàng xóm là đất?
- `union` gắn **gốc vào gốc** (`parent[rb] = ra`), có union by size/rank để cây thấp?
- `find` viết bằng vòng lặp (hoặc đã chắc cây thấp nếu dùng đệ quy)?
- Phân biệt được ô **nước** (`-1`) với ô đất là **gốc** (`parent[x] == x`)?
- Kết quả có đúng **`len(positions)`** phần tử không?
- Đã chạy: Example 1, Example 2, ô trùng, `1x1`, một hàng, một cột, chữ thập, chỉ chạm chéo, bẫy tràn hàng, bẫy chỉ số âm?